# Dynamic ORB Strategy
A stratey that chooses the best ORB TF to trade utilizing forest tree classifiers and time series

**Tested on a 16 year OHLCV /NQ data set**

## Module.1
### Dependencies

In [38]:
from pathlib import Path
from typing import Optional

import sklearn as sk
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf


## Module.2
### Data importing and resampling

In [56]:
# ═════════════════════════════════════════════════════════════════════════════
# IMPORT + RESAMPLE — NQ futures + 10 ETF 1-minute files
#
# Loads each 1-minute CSV, parses the timestamp, localizes to
# America/New_York, and resamples to 3m / 5m / 15m / 30m / 45m / 1h.
#
# Output: frames[asset][timeframe] = DataFrame
#   keys: 'NQ', 'CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT'
#   timeframes: '1m', '3m', '5m', '15m', '30m', '45m', '1h'
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
from pathlib import Path
from typing import Optional
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ───────────────────────────────────────────────────────────
NQ_DIR = Path("/Users/andrew/main/trading/data/NQ.1m.OHLCV.data")
ETF_FILES = {
    "CPER": Path("/Users/andrew/main/trading/data/etfs/CPER_clean_1min.csv"),
    "DIA":  Path("/Users/andrew/main/trading/data/etfs/DIA_clean_1min.csv"),
    "GLD":  Path("/Users/andrew/main/trading/data/etfs/GLD_clean_1min.csv"),
    "IEF":  Path("/Users/andrew/main/trading/data/etfs/IEF_clean_1min.csv"),
    "IWM":  Path("/Users/andrew/main/trading/data/etfs/IWM_clean_1min.csv"),
    "QQQ":  Path("/Users/andrew/main/trading/data/etfs/QQQ_clean_1min.csv"),
    "SHY":  Path("/Users/andrew/main/trading/data/etfs/SHY_clean_1min.csv"),
    "SLV":  Path("/Users/andrew/main/trading/data/etfs/SLV_clean_1min.csv"),
    "SPY":  Path("/Users/andrew/main/trading/data/etfs/SPY_clean_1min.csv"),
    "TLT":  Path("/Users/andrew/main/trading/data/etfs/TLT_clean_1min.csv"),
}

TZ = "America/New_York"
TARGETS: dict[str, str] = {
    "3m":  "3min",
    "5m":  "5min",
    "15m": "15min",
    "30m": "30min",
    "45m": "45min",
    "1h":  "1h",
}
AGG = {"open": "first", "high": "max", "low": "min",
       "close": "last", "volume": "sum"}


# ── Loader ──────────────────────────────────────────────────────────────────
def find_nq_csv(directory: Path) -> Path:
    """Return the CSV inside NQ.1m.OHLCV.data/ (handles multiple files)."""
    if not directory.exists():
        raise FileNotFoundError(f"NQ directory not found: {directory}")
    if directory.is_file() and directory.suffix.lower() == ".csv":
        return directory
    csvs = sorted(directory.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No .csv files inside {directory}")
    # Prefer the longest-history file if multiple exist
    return csvs[-1]


def load_ohlcv(filepath: Path, tz: Optional[str] = None) -> pd.DataFrame:
    """
    Load an OHLCV CSV into a tidy, tz-aware, DatetimeIndexed DataFrame.

    Handles:
      - combined 'datetime'/'timestamp'/'ts_event' column
      - separate 'date' + 'time' columns
      - pandas unnamed index column
      - case-insensitive header names
      - optional tz localization with DST-safe NaT handling
    """
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(filepath)

    raw = pd.read_csv(filepath)
    raw = raw.loc[:, ~raw.columns.str.match(r"^Unnamed: \d+$")]
    lower = {c.lower(): c for c in raw.columns}

    # ── Resolve datetime column ──────────────────────────────────────────
    dt_candidates = ["datetime", "date_time", "timestamp", "ts_event",
                     "ts_recv", "ts", "time"]
    dt_col = next((lower[c] for c in dt_candidates if c in lower), None)

    if dt_col is None:
        date_col = lower.get("date")
        time_col = lower.get("time")
        if date_col and time_col:
            raw["__dt__"] = raw[date_col].astype(str) + " " + raw[time_col].astype(str)
            dt_col = "__dt__"
        elif date_col:
            dt_col = date_col
        else:
            raise ValueError(f"No datetime column. Headers: {list(raw.columns)}")

    # ── Resolve OHLCV columns ────────────────────────────────────────────
    def pick(names):
        return next((lower[n] for n in names if n in lower), None)

    cols = {
        "open":   pick(["open", "o"]),
        "high":   pick(["high", "h"]),
        "low":    pick(["low", "l"]),
        "close":  pick(["close", "c", "last", "price"]),
        "volume": pick(["volume", "vol", "v", "size"]),
    }
    if any(v is None for v in cols.values()):
        raise ValueError(f"Unresolved columns: {cols}. Headers: {list(raw.columns)}")

    df = raw[[dt_col, *cols.values()]].copy()
    df.columns = ["datetime", *cols.keys()]

    # ── Parse datetime ───────────────────────────────────────────────────
    dt = pd.to_datetime(df["datetime"], utc=False, errors="coerce")
    if getattr(dt.dt, "tz", None) is None and tz:
        dt = dt.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    elif getattr(dt.dt, "tz", None) is not None and tz:
        dt = dt.dt.tz_convert(tz)

    df["datetime"] = dt
    df = df.dropna(subset=["datetime"]).set_index("datetime").sort_index()

    # ── Numerics ─────────────────────────────────────────────────────────
    for c in ("open", "high", "low", "close", "volume"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["open", "high", "low", "close"])
    df = df[~df.index.duplicated(keep="last")]
    return df


# ── Resampler ───────────────────────────────────────────────────────────────
def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """Resample a tz-aware OHLCV frame; drop empty buckets."""
    out = (
        df.resample(rule, label="left", closed="left", origin="start_day")
          .agg(AGG)
          .dropna(subset=["open", "high", "low", "close"])
    )
    # Drop bars where a full hour of volume is 0 (session gaps)
    return out


# ═════════════════════════════════════════════════════════════════════════════
# LOAD EVERYTHING
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("LOADING 1-MINUTE DATA")
print("=" * 80)

raw_1m: dict[str, pd.DataFrame] = {}

# NQ
nq_path = find_nq_csv(NQ_DIR)
print(f"  {'NQ':>5}  ← {nq_path.name}")
raw_1m["NQ"] = load_ohlcv(nq_path, tz=TZ)
print(f"         {len(raw_1m['NQ']):,} bars   "
      f"{raw_1m['NQ'].index.min()} → {raw_1m['NQ'].index.max()}")

# ETFs
for label, path in ETF_FILES.items():
    try:
        df = load_ohlcv(path, tz=TZ)
        raw_1m[label] = df
        print(f"  {label:>5}  ← {path.name}")
        print(f"         {len(df):,} bars   {df.index.min()} → {df.index.max()}")
    except Exception as e:
        print(f"  {label:>5}  FAILED: {type(e).__name__}: {e}")

print(f"\nLoaded {len(raw_1m)} assets at 1-minute resolution.\n")


# ═════════════════════════════════════════════════════════════════════════════
# RESAMPLE
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("RESAMPLING TO 3m / 5m / 15m / 30m / 45m / 1h")
print("=" * 80)

frames: dict[str, dict[str, pd.DataFrame]] = {}

for label, df_1m in raw_1m.items():
    frames[label] = {"1m": df_1m}
    for tf_name, rule in TARGETS.items():
        try:
            frames[label][tf_name] = resample_ohlcv(df_1m, rule)
        except Exception as e:
            print(f"  {label} → {tf_name} FAILED: {type(e).__name__}: {e}")

print(f"Built frames for {len(frames)} assets × "
      f"{len(next(iter(frames.values())))} timeframes.\n")


# ═════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═════════════════════════════════════════════════════════════════════════════
rows = []
for label, tfs in frames.items():
    for tf_name, d in tfs.items():
        rows.append({
            "asset":     label,
            "timeframe": tf_name,
            "bars":      len(d),
            "start":     d.index.min() if len(d) else pd.NaT,
            "end":       d.index.max() if len(d) else pd.NaT,
        })

summary = pd.DataFrame(rows)

# Pivot for a readable per-asset view
pivot = summary.pivot(index="asset", columns="timeframe", values="bars") \
               .reindex(columns=["1m", "3m", "5m", "15m", "30m", "45m", "1h"])

print("=" * 80)
print("BAR COUNT PER ASSET × TIMEFRAME")
print("=" * 80)
display(pivot)

print("\nDate coverage per asset (1-minute frame):")
coverage = summary[summary["timeframe"] == "1m"][["asset", "start", "end"]] \
    .set_index("asset")
display(coverage)

# ── Access patterns (for later cells) ───────────────────────────────────────
print("\nAccess patterns:")
print("  frames['NQ']['5m']        → NQ 5-minute bars")
print("  frames['SPY']['1h']       → SPY hourly bars")
print("  frames['QQQ']['1m']       → QQQ 1-minute bars")
print(f"\nAvailable assets:     {sorted(frames.keys())}")
print(f"Available timeframes: {list(next(iter(frames.values())).keys())}")

LOADING 1-MINUTE DATA
     NQ  ← NQ-2010.06.06-2026.01.23-ohlcv-1m.csv
         5,293,774 bars   2010-06-06 18:00:00-04:00 → 2026-01-23 16:59:00-05:00
   CPER  ← CPER_clean_1min.csv
         109,059 bars   2011-11-15 09:37:00-05:00 → 2026-09-21 15:59:00-04:00
    DIA  ← DIA_clean_1min.csv
         2,067,170 bars   2002-12-30 09:30:00-05:00 → 2026-09-21 15:58:00-04:00
    GLD  ← GLD_clean_1min.csv
         1,924,936 bars   2004-11-18 09:30:00-05:00 → 2026-09-21 15:59:00-04:00
    IEF  ← IEF_clean_1min.csv
         1,560,831 bars   2002-12-30 09:31:00-05:00 → 2026-09-21 15:57:00-04:00
    IWM  ← IWM_clean_1min.csv
         2,217,669 bars   2002-12-30 09:30:00-05:00 → 2026-09-21 15:57:00-04:00
    QQQ  ← QQQ_clean_1min.csv
         2,245,015 bars   2002-12-30 09:30:00-05:00 → 2026-09-21 15:56:00-04:00
    SHY  ← SHY_clean_1min.csv
         1,523,666 bars   2002-12-30 09:30:00-05:00 → 2026-09-21 15:59:00-04:00
    SLV  ← SLV_clean_1min.csv
         1,752,691 bars   2006-04-28 09:32:00-04:0

timeframe,1m,3m,5m,15m,30m,45m,1h
asset,,,,,,,
CPER,109059,74972,59547,32918,21622,17621,14497
DIA,2067170,743508,455164,154200,77208,55571,41600
GLD,1924936,687909,419724,141901,71069,51117,38286
IEF,1560831,677440,433235,153120,77130,55518,41584
IWM,2217669,763287,461229,154450,77233,55576,41604
NQ,5293774,1821013,1097034,366257,184862,124477,93140
QQQ,2245015,765284,461850,154538,77275,55609,41624
SHY,1523666,657165,420809,151339,76899,55447,41551
SLV,1752691,639037,391333,132527,66373,49359,35758



Date coverage per asset (1-minute frame):


,start,end
asset,,
NQ,2010-06-06 18:00:00-04:00,2026-01-23 16:59:00-05:00
CPER,2011-11-15 09:37:00-05:00,2026-09-21 15:59:00-04:00
DIA,2002-12-30 09:30:00-05:00,2026-09-21 15:58:00-04:00
GLD,2004-11-18 09:30:00-05:00,2026-09-21 15:59:00-04:00
IEF,2002-12-30 09:31:00-05:00,2026-09-21 15:57:00-04:00
IWM,2002-12-30 09:30:00-05:00,2026-09-21 15:57:00-04:00
QQQ,2002-12-30 09:30:00-05:00,2026-09-21 15:56:00-04:00
SHY,2002-12-30 09:30:00-05:00,2026-09-21 15:59:00-04:00
SLV,2006-04-28 09:32:00-04:00,2026-09-21 15:59:00-04:00



Access patterns:
  frames['NQ']['5m']        → NQ 5-minute bars
  frames['SPY']['1h']       → SPY hourly bars
  frames['QQQ']['1m']       → QQQ 1-minute bars

Available assets:     ['CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'NQ', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT']
Available timeframes: ['1m', '3m', '5m', '15m', '30m', '45m', '1h']


## Module.3
### ORB Strategy implementation
**Bootstraps multiple variations to find the best variant**

In [60]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB BOOTSTRAP — LOCAL DATA, ALL ASSETS (NQ + 10 ETFs)
#
# Vectorized multi-asset ORB with walk-forward validation.
# Includes skip-reason accounting in both IS and OOS loops so we can see
# exactly why configs are being filtered out.
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
import itertools
import time
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ───────────────────────────────────────────────────────────
INITIAL_EQUITY   = 25_000.0
RISK_PER_TRADE   = 0.01
LEVERAGE_CAP     = 4.0
COMMISSION       = 0.0035
SLIPPAGE_TICKS   = 1
DEFAULT_TICK     = 0.01

RTH_START        = pd.Timestamp("09:30").time()
RTH_END          = pd.Timestamp("16:00").time()
REL_VOL_LOOKBACK = 14
ATR_LOOKBACK     = 14

MIN_PRICE        = 5.0
MIN_AVG_VOL      = 1_000_000
MIN_ATR          = 0.50

WF_TRAIN_FRAC   = 0.60
MIN_TRADES_IN   = 5      # lowered from 20; adjust upward once coverage is understood
MIN_TRADES_OOS  = 10     # lowered from 30
MAX_CONFIGS     = 800

OR_WINDOWS = [5, 15, 30, 45, 60]
BASE_TF    = "1m"

# ── Require `frames` from the loader cell ───────────────────────────────────
try:
    frames
    print(f"Using `frames` from prior cell ({len(frames)} assets).")
except NameError:
    raise RuntimeError(
        "`frames` not found. Run the data-loading cell first to populate it."
    )


# ═════════════════════════════════════════════════════════════════════════════
# PREPROCESS ONE ASSET
# ═════════════════════════════════════════════════════════════════════════════
def preprocess_asset(df: pd.DataFrame, tz: str = "America/New_York") -> dict:
    if df.index.tz is None:
        df = df.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    df = df.tz_convert(tz)
    times = df.index.time
    rth = (times >= RTH_START) & (times < RTH_END)
    df_rth = df[rth].copy()
    df_rth["date"] = df_rth.index.normalize().tz_localize(None)
    if df_rth.empty:
        return {}

    # Session VWAP + volume SMA
    tp = (df_rth["high"] + df_rth["low"] + df_rth["close"]) / 3
    df_rth["vwap"] = (
        (tp * df_rth["volume"]).groupby(df_rth["date"]).cumsum()
        / df_rth["volume"].groupby(df_rth["date"]).cumsum().replace(0, np.nan)
    )
    df_rth["vol_sma20"] = df_rth["volume"].rolling(20, min_periods=20).mean()

    # Daily aggregates
    AGG = {"open":"first","high":"max","low":"min","close":"last","volume":"sum"}
    daily = df_rth.resample("1D").agg(AGG).dropna(subset=["open","high","low","close"])
    if daily.index.tz is not None:
        daily.index = daily.index.tz_localize(None)

    prev_close = daily["close"].shift(1)
    tr = pd.concat([
        daily["high"] - daily["low"],
        (daily["high"] - prev_close).abs(),
        (daily["low"]  - prev_close).abs(),
    ], axis=1).max(axis=1)
    daily["atr14"]    = tr.ewm(alpha=1/ATR_LOOKBACK, adjust=False,
                               min_periods=ATR_LOOKBACK).mean()
    daily["atr_prev"] = daily["atr14"].shift(1)
    daily["sma20"]    = daily["close"].rolling(20, min_periods=20).mean()
    daily["range"]    = daily["high"] - daily["low"]
    daily["nr7"]      = daily["range"] == daily["range"].rolling(7, min_periods=7).min()
    daily["gap"]      = daily["open"] - prev_close
    daily["avgvol14"] = daily["volume"].rolling(14, min_periods=14).mean()

    # Per-OR-window caches
    or_cache: dict[int, pd.DataFrame] = {}
    exec_cache: dict[int, dict] = {}
    for orb_min in OR_WINDOWS:
        rows = []
        exec_cache[orb_min] = {}
        for date, sess in df_rth.groupby("date", sort=True):
            if len(sess) < orb_min + 1:
                continue
            or_bars = sess.iloc[:orb_min]
            o = or_bars["open"].iloc[0]
            c = or_bars["close"].iloc[-1]
            h = or_bars["high"].max()
            l = or_bars["low"].min()
            v = or_bars["volume"].sum()
            rows.append({"date": date, "or_open": o, "or_close": c,
                         "or_high": h, "or_low": l, "or_vol": v,
                         "or_range": h - l,
                         "direction": 1 if c > o else (-1 if c < o else 0)})
            exec_cache[orb_min][date] = sess.iloc[orb_min:]

        or_df = pd.DataFrame(rows).set_index("date")
        or_df["rel_vol"] = or_df["or_vol"] / (
            or_df["or_vol"].rolling(REL_VOL_LOOKBACK, min_periods=REL_VOL_LOOKBACK)
                    .mean().shift(1))
        for col in ("atr_prev","sma20","nr7","gap"):
            if col in daily.columns:
                or_df[col] = daily[col].reindex(or_df.index)
        or_cache[orb_min] = or_df

    return {"df_rth": df_rth, "daily": daily,
            "or_cache": or_cache, "exec_cache": exec_cache}


print("\nPreprocessing assets ...")
asset_data: dict[str, dict] = {}
for label, tfs in frames.items():
    base = tfs.get(BASE_TF)
    if base is None or base.empty:
        print(f"  {label:>5}: no {BASE_TF} frame — skipped")
        continue
    info = preprocess_asset(base)
    if info:
        asset_data[label] = info
        n_sess = info["df_rth"]["date"].nunique()
        print(f"  {label:>5}: {len(info['df_rth']):,} RTH bars, {n_sess:,} sessions")

if not asset_data:
    raise RuntimeError("No assets preprocessed successfully.")

print(f"\nPreprocessed {len(asset_data)} assets.\n")


# ═════════════════════════════════════════════════════════════════════════════
# VECTORIZED MULTI-ASSET ORB BACKTEST
# ═════════════════════════════════════════════════════════════════════════════
def backtest_portfolio(
    asset_data: dict,
    orb_min: int,
    direction_source: str,
    or_size_filter: tuple[float, float],
    use_nr7: bool,
    rel_vol_min: float,
    top_n: int,
    breakout_vol_mult: float,
    use_vwap_filter: bool,
    require_retest: bool,
    stop_type: str,
    stop_param: float,
    target_type: str,
    target_param: float,
    entry_type: str,
    date_range: tuple | None = None,
    tick: float = DEFAULT_TICK,
):
    lo_f, hi_f = or_size_filter
    signal_frames: list[pd.DataFrame] = []

    # ── 1. Vectorized signal construction per asset ─────────────────────
    for asset, info in asset_data.items():
        or_df = info["or_cache"].get(orb_min)
        if or_df is None or or_df.empty:
            continue
        daily = info["daily"]

        if date_range is not None:
            d_lo, d_hi = date_range
            or_df = or_df[(or_df.index >= d_lo) & (or_df.index <= d_hi)]
            if or_df.empty:
                continue

        common = or_df.index.intersection(daily.index)
        if len(common) == 0:
            continue
        or_df = or_df.loc[common]
        dsub  = daily.loc[common]

        atr      = or_df["atr_prev"].to_numpy(dtype=np.float64)
        or_range = or_df["or_range"].to_numpy(dtype=np.float64)

        with np.errstate(divide="ignore", invalid="ignore"):
            or_atr = np.where(atr > 0, or_range / atr, np.nan)

        valid = np.isfinite(atr) & (atr > 0)
        valid &= np.isfinite(or_atr) & (or_atr >= lo_f) & (or_atr <= hi_f)

        # Preserve original semantics: NaN nr7 passes the filter (bool(NaN) is True)
        if use_nr7:
            valid &= or_df["nr7"].fillna(True).to_numpy(dtype=bool)

        if rel_vol_min > 0:
            rv = or_df["rel_vol"].to_numpy(dtype=np.float64)
            valid &= np.isfinite(rv) & (rv >= rel_vol_min)

        # Paper's stock-universe filters
        d_open = dsub["open"].to_numpy(dtype=np.float64)
        d_avgvol = (dsub["avgvol14"].to_numpy(dtype=np.float64)
                    if "avgvol14" in dsub.columns
                    else np.full(len(dsub), np.nan))
        valid &= np.isfinite(d_open)   & (d_open   >= MIN_PRICE)
        valid &= np.isfinite(d_avgvol) & (d_avgvol >= MIN_AVG_VOL)
        valid &= (atr >= MIN_ATR)

        if not valid.any():
            continue

        sub = or_df.loc[valid]
        dsub_valid = dsub.loc[valid]

        sig = pd.DataFrame({
            "asset":            asset,
            "date":             sub.index.values,
            "direction_first":  sub["direction"].astype(int).to_numpy(),
            "or_high":          sub["or_high"].to_numpy(dtype=np.float64),
            "or_low":           sub["or_low"].to_numpy(dtype=np.float64),
            "or_range":         sub["or_range"].to_numpy(dtype=np.float64),
            "or_atr":           or_atr[valid],
            "atr":              sub["atr_prev"].to_numpy(dtype=np.float64),
            "gap":              sub["gap"].to_numpy(dtype=np.float64),
            "sma20":            sub["sma20"].to_numpy(dtype=np.float64),
            "rel_vol":          sub["rel_vol"].to_numpy(dtype=np.float64),
            "day_close":        dsub_valid["close"].to_numpy(dtype=np.float64),
        })
        signal_frames.append(sig)

    if not signal_frames:
        return pd.DataFrame(), pd.Series(dtype=float)

    sig_df = pd.concat(signal_frames, ignore_index=True)
    sig_df["date"] = pd.to_datetime(sig_df["date"])
    sig_df["rv_rank"] = sig_df.groupby("date")["rel_vol"].rank(
        ascending=False, method="first")
    sig_df = sig_df[sig_df["rv_rank"] <= top_n]
    if sig_df.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    # ── 2. Trade simulation (sequential equity; vectorized inner scans) ─
    equity = INITIAL_EQUITY
    trades: list[dict] = []
    eq_hist: dict[pd.Timestamp, float] = {}

    for date, day_sigs in sig_df.groupby("date", sort=True):
        date_ts = pd.Timestamp(date)
        eq_hist[date_ts] = equity

        for sig in day_sigs.itertuples(index=False):
            asset = sig.asset
            info = asset_data[asset]
            exec_bars = info["exec_cache"].get(orb_min, {}).get(date_ts)
            if exec_bars is None or exec_bars.empty:
                continue

            first_dir = int(sig.direction_first)
            gap = float(sig.gap)
            sma20 = float(sig.sma20)
            day_close = float(sig.day_close)

            if direction_source == "first_bar":
                direction = first_dir
            elif direction_source == "gap":
                if not np.isfinite(gap) or gap == 0:
                    continue
                direction = 1 if gap > 0 else -1
                if direction != first_dir:
                    continue
            elif direction_source == "sma_trend":
                if not np.isfinite(sma20) or not np.isfinite(day_close):
                    continue
                direction = 1 if day_close > sma20 else -1
                if direction != first_dir:
                    continue
            else:
                raise ValueError(direction_source)

            if direction == 0:
                continue

            trigger = float(sig.or_high) if direction == 1 else float(sig.or_low)

            high_a   = exec_bars["high"].to_numpy(dtype=np.float64)
            low_a    = exec_bars["low"].to_numpy(dtype=np.float64)
            open_a   = exec_bars["open"].to_numpy(dtype=np.float64)
            close_a  = exec_bars["close"].to_numpy(dtype=np.float64)
            volume_a = exec_bars["volume"].to_numpy(dtype=np.float64)

            # ── Trigger detection ───────────────────────────────────────
            if direction == 1:
                hits = high_a >= trigger
            else:
                hits = low_a <= trigger
            if not hits.any():
                continue
            hit_idx = int(hits.argmax())

            # ── Breakout volume filter ──────────────────────────────────
            if breakout_vol_mult > 0:
                if "vol_sma20" not in exec_bars.columns:
                    continue
                vsma_arr = exec_bars["vol_sma20"].to_numpy(dtype=np.float64)
                if not np.isfinite(vsma_arr[hit_idx]):
                    continue
                if volume_a[hit_idx] < breakout_vol_mult * vsma_arr[hit_idx]:
                    continue

            # ── Entry ───────────────────────────────────────────────────
            if entry_type == "stop":
                bar_open = open_a[hit_idx]
                raw_fill = max(trigger, bar_open) if direction == 1 \
                    else min(trigger, bar_open)
                entry_px = raw_fill + direction * SLIPPAGE_TICKS * tick
            elif entry_type == "close":
                entry_px = close_a[hit_idx] + direction * SLIPPAGE_TICKS * tick
            else:
                raise ValueError(entry_type)
            entry_bar_idx = hit_idx

            # ── VWAP filter ─────────────────────────────────────────────
            if use_vwap_filter:
                if "vwap" not in exec_bars.columns:
                    continue
                vwap_arr = exec_bars["vwap"].to_numpy(dtype=np.float64)
                if not np.isfinite(vwap_arr[hit_idx]):
                    continue
                if direction == 1 and entry_px < vwap_arr[hit_idx]:
                    continue
                if direction == -1 and entry_px > vwap_arr[hit_idx]:
                    continue

            # ── Retest (vectorized) ─────────────────────────────────────
            if require_retest:
                if entry_bar_idx + 1 >= len(exec_bars):
                    continue
                post_low   = low_a[entry_bar_idx + 1:]
                post_high  = high_a[entry_bar_idx + 1:]
                post_close = close_a[entry_bar_idx + 1:]
                if direction == 1:
                    retest_mask = (post_low <= trigger) & (post_close > trigger)
                else:
                    retest_mask = (post_high >= trigger) & (post_close < trigger)
                if not retest_mask.any():
                    continue
                retest_pos = int(retest_mask.argmax())
                entry_px = post_close[retest_pos] + direction * SLIPPAGE_TICKS * tick
                entry_bar_idx = entry_bar_idx + 1 + retest_pos

            # ── Stop ────────────────────────────────────────────────────
            atr = float(sig.atr)
            if stop_type == "atr":
                stop_dist = stop_param * atr
            elif stop_type == "or_opposite":
                stop_dist = stop_param * float(sig.or_range)
            elif stop_type == "fixed":
                stop_dist = stop_param
            else:
                raise ValueError(stop_type)
            if stop_dist <= 0:
                continue
            stop_px = entry_px - direction * stop_dist

            # ── Sizing ──────────────────────────────────────────────────
            risk_dollars = RISK_PER_TRADE * equity
            shares = risk_dollars / stop_dist
            if shares * entry_px > LEVERAGE_CAP * equity:
                shares = LEVERAGE_CAP * equity / entry_px
            if shares <= 0 or not np.isfinite(shares):
                continue

            # ── Target ──────────────────────────────────────────────────
            if target_type == "eod":
                target_px = np.nan
            elif target_type == "or_mult":
                target_px = entry_px + direction * target_param * float(sig.or_range)
            elif target_type == "R_mult":
                target_px = entry_px + direction * target_param * stop_dist
            else:
                raise ValueError(target_type)

            # ── Exit scan (vectorized) ──────────────────────────────────
            if entry_bar_idx + 1 >= len(exec_bars):
                exit_px = close_a[entry_bar_idx]
                exit_reason = "eod"
            else:
                after_high = high_a[entry_bar_idx + 1:]
                after_low  = low_a[entry_bar_idx + 1:]
                n = len(after_high)

                if direction == 1:
                    stop_hits = after_low <= stop_px
                else:
                    stop_hits = after_high >= stop_px

                if not np.isnan(target_px):
                    if direction == 1:
                        tgt_hits = after_high >= target_px
                    else:
                        tgt_hits = after_low <= target_px
                    tgt_pos = int(tgt_hits.argmax()) if tgt_hits.any() else n
                else:
                    tgt_pos = n

                stop_pos = int(stop_hits.argmax()) if stop_hits.any() else n

                # Target wins ties (original checks target before stop per bar)
                if tgt_pos < n and tgt_pos <= stop_pos:
                    exit_px = target_px - direction * SLIPPAGE_TICKS * tick
                    exit_reason = "target"
                elif stop_pos < n:
                    exit_px = stop_px - direction * SLIPPAGE_TICKS * tick
                    exit_reason = "stop"
                else:
                    exit_px = close_a[-1]
                    exit_reason = "eod"

            # ── PnL ─────────────────────────────────────────────────────
            gross = direction * (exit_px - entry_px) * shares
            comm  = COMMISSION * shares * 2
            net   = gross - comm
            R     = direction * (exit_px - entry_px) / stop_dist

            equity += net
            trades.append({
                "date": date_ts, "asset": asset, "direction": direction,
                "rel_vol": sig.rel_vol, "or_atr": sig.or_atr,
                "trigger": trigger, "entry_px": entry_px,
                "stop_px": stop_px, "exit_px": exit_px,
                "exit_reason": exit_reason,
                "shares": shares, "net_pnl": net, "R": R,
            })

        eq_hist[date_ts] = equity

    # ── 3. Build equity series with guaranteed DatetimeIndex ────────────
    eq_series = pd.Series(eq_hist, dtype=np.float64)
    if not isinstance(eq_series.index, pd.DatetimeIndex):
        eq_series.index = pd.to_datetime(eq_series.index)
    eq_series = eq_series.sort_index()
    eq_series.name = "equity"

    return pd.DataFrame(trades), eq_series


# ═════════════════════════════════════════════════════════════════════════════
# PARAMETER GRID
# ═════════════════════════════════════════════════════════════════════════════
GRID = {
    "orb_min":           [5, 15, 30, 45, 60],
    "direction_source":  ["first_bar", "gap", "sma_trend"],
    "or_size_filter":    [(0, np.inf), (0.3, 1.5), (0.5, 1.2), (0.0, 0.6)],
    "use_nr7":           [False, True],
    "rel_vol_min":       [0.0, 1.0, 1.5, 2.0],
    "top_n":             [3, 5, 10],
    "breakout_vol_mult": [0.0, 1.2, 1.5],
    "use_vwap_filter":   [False, True],
    "require_retest":    [False, True],
    "stop_type":         ["atr"],
    "stop_param":        [0.10, 0.25, 0.50],
    "target_type":       ["eod", "or_mult", "R_mult"],
    "target_param":      [0.5, 1.0, 1.5, 3.0],
    "entry_type":        ["stop", "close"],
}
keys = list(GRID.keys())
combos = list(itertools.product(*[GRID[k] for k in keys]))
valid = []
for c in combos:
    d = dict(zip(keys, c))
    if d["target_type"] == "eod" and d["target_param"] != 0.5:
        continue
    if d["stop_type"] == "fixed":
        continue
    valid.append(d)

print(f"Grid: {len(combos):,} raw → {len(valid):,} valid")
if len(valid) > MAX_CONFIGS:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(valid), MAX_CONFIGS, replace=False)
    valid = [valid[i] for i in idx]
    print(f"Stratified sample: {len(valid):,} configs")


# ═════════════════════════════════════════════════════════════════════════════
# IN-SAMPLE / OUT-OF-SAMPLE SPLIT
# ═════════════════════════════════════════════════════════════════════════════
all_dates = sorted(set().union(*[
    info["or_cache"][5].index
    for info in asset_data.values()
    if 5 in info["or_cache"]
]))
split_idx  = int(len(all_dates) * WF_TRAIN_FRAC)
split_date = all_dates[split_idx]
in_range   = (all_dates[0], split_date)
oos_range  = (split_date + pd.Timedelta(days=1), all_dates[-1])

print(f"\nIS ends {split_date.date()}  |  OOS starts {oos_range[0].date()}\n")


# ═════════════════════════════════════════════════════════════════════════════
# IN-SAMPLE GRID SEARCH (with skip-reason accounting)
# ═════════════════════════════════════════════════════════════════════════════
print("In-sample grid search ...")
results = []
skips = {"exception": 0, "no_signals": 0, "few_trades": 0, "no_variance": 0}
exception_samples: list[str] = []
t0 = time.time()

for i, params in enumerate(valid):
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(valid)}  ({time.time()-t0:.0f}s)")

    try:
        trades, equity = backtest_portfolio(asset_data, **params, date_range=in_range)
    except Exception as e:
        skips["exception"] += 1
        if len(exception_samples) < 5:
            exception_samples.append(f"{type(e).__name__}: {e} | params={params}")
        continue

    if trades.empty:
        skips["no_signals"] += 1
        continue
    if len(trades) < MIN_TRADES_IN:
        skips["few_trades"] += 1
        continue

    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        skips["no_variance"] += 1
        continue

    results.append({
        **{k: v for k, v in params.items() if k != "or_size_filter"},
        "or_size_lo": params["or_size_filter"][0],
        "or_size_hi": params["or_size_filter"][1],
        "in_trades":  len(trades),
        "in_sharpe":  float(rets.mean() / rets.std() * np.sqrt(252)),
        "in_return":  float(equity.iloc[-1] / INITIAL_EQUITY - 1),
        "in_hit":     float((trades["R"] > 0).mean()),
        "in_avgR":    float(trades["R"].mean()),
    })

res_df = pd.DataFrame(results)
if res_df.empty:
    raise RuntimeError(f"Zero configs survived IS. Skip counts: {skips}")

res_df = res_df.sort_values("in_sharpe", ascending=False).reset_index(drop=True)

print(f"\nIn-sample summary:")
print(f"  evaluated:              {len(res_df):,} / {len(valid):,}")
print(f"  skipped — exception:    {skips['exception']}")
print(f"  skipped — no signals:   {skips['no_signals']}")
print(f"  skipped — few trades:   {skips['few_trades']}")
print(f"  skipped — no variance:  {skips['no_variance']}")
if exception_samples:
    print("\nFirst exceptions:")
    for s in exception_samples:
        print(f"  {s}")

print("\n" + "=" * 90)
print("TOP 15 IN-SAMPLE CONFIGS")
print("=" * 90)
show_cols = ["orb_min","direction_source","rel_vol_min","top_n",
             "breakout_vol_mult","use_vwap_filter","require_retest",
             "stop_param","target_type","target_param","entry_type",
             "in_trades","in_sharpe","in_return","in_hit","in_avgR"]
display(res_df[show_cols].head(15).round(3))


# ═════════════════════════════════════════════════════════════════════════════
# OUT-OF-SAMPLE VALIDATION (with skip-reason accounting)
# ═════════════════════════════════════════════════════════════════════════════
print("\nOut-of-sample validation on top 50 IS configs ...")
oos_results = []
oos_skips = {"exception": 0, "no_signals": 0, "few_trades": 0, "no_variance": 0}

for _, row in res_df.head(50).iterrows():
    params = {
        "orb_min":           int(row["orb_min"]),
        "direction_source":  row["direction_source"],
        "or_size_filter":    (float(row["or_size_lo"]), float(row["or_size_hi"])),
        "use_nr7":           bool(row["use_nr7"]),
        "rel_vol_min":       float(row["rel_vol_min"]),
        "top_n":             int(row["top_n"]),
        "breakout_vol_mult": float(row["breakout_vol_mult"]),
        "use_vwap_filter":   bool(row["use_vwap_filter"]),
        "require_retest":    bool(row["require_retest"]),
        "stop_type":         row["stop_type"],
        "stop_param":        float(row["stop_param"]),
        "target_type":       row["target_type"],
        "target_param":      float(row["target_param"]),
        "entry_type":        row["entry_type"],
    }
    try:
        trades, equity = backtest_portfolio(asset_data, **params, date_range=oos_range)
    except Exception as e:
        oos_skips["exception"] += 1
        print(f"    OOS EXCEPTION: {type(e).__name__}: {e}")
        continue

    if trades.empty:
        oos_skips["no_signals"] += 1
        continue
    if len(trades) < MIN_TRADES_OOS:
        oos_skips["few_trades"] += 1
        continue

    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        oos_skips["no_variance"] += 1
        continue

    oos_results.append({
        **params, "or_size_filter": str(params["or_size_filter"]),
        "oos_trades": len(trades),
        "oos_sharpe": float(rets.mean() / rets.std() * np.sqrt(252)),
        "oos_return": float(equity.iloc[-1] / INITIAL_EQUITY - 1),
        "oos_hit":    float((trades["R"] > 0).mean()),
        "oos_avgR":   float(trades["R"].mean()),
    })

print(f"\nOOS summary:")
print(f"  evaluated:              {len(oos_results):,} / 50")
print(f"  skipped — exception:    {oos_skips['exception']}")
print(f"  skipped — no signals:   {oos_skips['no_signals']}")
print(f"  skipped — few trades:   {oos_skips['few_trades']}")
print(f"  skipped — no variance:  {oos_skips['no_variance']}")

oos_df = pd.DataFrame(oos_results)
if oos_df.empty:
    raise RuntimeError(
        f"Zero configs survived OOS. Skips: {oos_skips}. "
        "Lower MIN_TRADES_OOS or expand the top-50 cut."
    )
oos_df = oos_df.sort_values("oos_sharpe", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("TOP 15 OUT-OF-SAMPLE CONFIGS")
print("=" * 90)
show_oos = ["orb_min","direction_source","rel_vol_min","top_n",
            "use_vwap_filter","require_retest","stop_param",
            "target_type","target_param","entry_type",
            "oos_trades","oos_sharpe","oos_return","oos_hit","oos_avgR"]
display(oos_df[show_oos].head(15).round(3))


# ═════════════════════════════════════════════════════════════════════════════
# BEST CONFIG → FULL SAMPLE
# ═════════════════════════════════════════════════════════════════════════════
best = oos_df.iloc[0]
print("\n" + "=" * 90); print("BEST CONFIG"); print("=" * 90)
for k, v in best.items():
    if k != "or_size_filter":
        print(f"  {k:>20}: {v}")

best_params = {
    "orb_min":           int(best["orb_min"]),
    "direction_source":  best["direction_source"],
    "or_size_filter":    eval(best["or_size_filter"]),
    "use_nr7":           bool(best["use_nr7"]),
    "rel_vol_min":       float(best["rel_vol_min"]),
    "top_n":             int(best["top_n"]),
    "breakout_vol_mult": float(best["breakout_vol_mult"]),
    "use_vwap_filter":   bool(best["use_vwap_filter"]),
    "require_retest":    bool(best["require_retest"]),
    "stop_type":         best["stop_type"],
    "stop_param":        float(best["stop_param"]),
    "target_type":       best["target_type"],
    "target_param":      float(best["target_param"]),
    "entry_type":        best["entry_type"],
}

trades_full, equity_full = backtest_portfolio(asset_data, **best_params, date_range=None)


# ═════════════════════════════════════════════════════════════════════════════
# PERFORMANCE METRICS
# ═════════════════════════════════════════════════════════════════════════════
def perf_stats(equity, trades, name):
    eq = equity.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if eq.empty or len(eq) < 2:
        return {"strategy": name}
    rets = eq.pct_change(fill_method=None).dropna()
    years = len(eq) / 252.0
    total = eq.iloc[-1] / INITIAL_EQUITY - 1
    cagr = (eq.iloc[-1] / INITIAL_EQUITY) ** (1 / years) - 1 \
        if years > 0 and eq.iloc[-1] > 0 else np.nan
    vol = rets.std() * np.sqrt(252) if len(rets) else np.nan
    sharpe = (rets.mean() * 252) / vol if vol and vol > 0 else np.nan
    mdd = (eq / eq.cummax() - 1).min()
    hit = (trades["R"] > 0).mean() if len(trades) else np.nan
    avgR = trades["R"].mean() if len(trades) else np.nan
    return {"strategy": name, "trades": len(trades),
            "total_return_%": total * 100, "CAGR_%": cagr * 100,
            "vol_%": vol * 100, "sharpe": sharpe,
            "hit_%": hit * 100, "avgR": avgR, "MDD_%": mdd * 100}

full_stats = perf_stats(equity_full, trades_full, "BEST ORB Portfolio")
print("\n" + "=" * 90); print("FULL-SAMPLE PERFORMANCE"); print("=" * 90)
display(pd.DataFrame([full_stats]).set_index("strategy").round(3))


# ═════════════════════════════════════════════════════════════════════════════
# BUY & HOLD (equal-weight) + EQUITY PLOT
# ═════════════════════════════════════════════════════════════════════════════
all_closes: dict[str, pd.Series] = {}
for asset, info in asset_data.items():
    close = info["df_rth"].groupby("date")["close"].last()
    if len(close) > 1:
        all_closes[asset] = close

close_df = pd.DataFrame(all_closes).sort_index()

normalized = close_df.copy()
for col in normalized.columns:
    valid = normalized[col].dropna()
    if valid.empty:
        continue
    normalized[col] = normalized[col] / valid.iloc[0]

bh_equity = (normalized.mean(axis=1) * INITIAL_EQUITY).dropna()
bh_equity.name = "Buy & Hold"

common_idx = equity_full.index.intersection(bh_equity.index)
if len(common_idx) == 0:
    raise RuntimeError("Strategy and buy & hold share no common dates.")

eq_plot = equity_full.reindex(common_idx)
bh_plot = bh_equity.reindex(common_idx)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bh_plot.index, y=bh_plot.values,
    name="Buy & Hold (equal-weight, {} assets)".format(len(asset_data)),
    line=dict(color="red", width=1.5, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=eq_plot.index, y=eq_plot.values,
    name=(f"BEST ORB ({best['orb_min']}m, {best['direction_source']}, "
          f"top-{best['top_n']} RV, "
          f"{'retest' if best['require_retest'] else 'direct'}, "
          f"stop {best['stop_param']}×ATR, "
          f"target {best['target_type']}({best['target_param']}))"),
    line=dict(color="blue", width=2.5),
))
fig.update_layout(
    title=(f"ORB Bootstrap — Local 1-minute data across {len(asset_data)} assets"
           f"<br><sup>Cross-sectional top-N Relative Volume selection · "
           f"paper stock-universe filters applied pre-ranking</sup>"),
    xaxis_title="Date", yaxis_title="Equity ($)",
    template="plotly_white", height=700, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(tickformat="$,.0f")
fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS
# ═════════════════════════════════════════════════════════════════════════════
if not trades_full.empty:
    print("\nExit-reason split:")
    display(trades_full["exit_reason"].value_counts())
    print("\nR distribution by exit reason:")
    display(trades_full.groupby("exit_reason")["R"]
                    .agg(["count", "mean", "std"]).round(3))
    print("\nDirection split (1=long, -1=short):")
    display(trades_full["direction"].value_counts())
    print("\nPer-asset trade count and PnL:")
    display(trades_full.groupby("asset").agg(
        trades=("net_pnl", "count"),
        total_pnl=("net_pnl", "sum"),
        avgR=("R", "mean"),
        hit=("R", lambda s: (s > 0).mean()),
    ).round(3).sort_values("total_pnl", ascending=False))

print(f"\n✓ Bootstrap complete.")
print(f"  Assets:                     {len(asset_data)}")
print(f"  IS configs evaluated:       {len(res_df):,}")
print(f"  OOS configs evaluated:      {len(oos_df):,}")
print(f"  Best OOS Sharpe:            {best['oos_sharpe']:.3f}")
print(f"  Best full-sample Sharpe:    {full_stats.get('sharpe', np.nan):.3f}")
print(f"  Best full-sample return:    {full_stats.get('total_return_%', np.nan):.1f}%")

Using `frames` from prior cell (11 assets).

Preprocessing assets ...
     NQ: 1,542,462 RTH bars, 4,016 sessions
   CPER: 109,059 RTH bars, 3,315 sessions
    DIA: 2,067,170 RTH bars, 5,966 sessions
    GLD: 1,924,936 RTH bars, 5,490 sessions
    IEF: 1,560,831 RTH bars, 5,966 sessions
    IWM: 2,217,669 RTH bars, 5,966 sessions
    QQQ: 2,245,015 RTH bars, 5,968 sessions
    SHY: 1,523,666 RTH bars, 5,966 sessions
    SLV: 1,752,691 RTH bars, 5,128 sessions
    SPY: 2,250,959 RTH bars, 5,968 sessions
    TLT: 2,005,004 RTH bars, 5,966 sessions

Preprocessed 11 assets.

Grid: 1,244,160 raw → 933,120 valid
Stratified sample: 800 configs

IS ends 2017-04-21  |  OOS starts 2017-04-22

In-sample grid search ...
  100/800  (65s)
  200/800  (115s)
  300/800  (143s)
  400/800  (195s)
  500/800  (224s)
  600/800  (255s)
  700/800  (290s)
  800/800  (321s)

In-sample summary:
  evaluated:              724 / 800
  skipped — exception:    0
  skipped — no signals:   28
  skipped — few trades:   

,orb_min,direction_source,rel_vol_min,top_n,breakout_vol_mult,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,in_trades,in_sharpe,in_return,in_hit,in_avgR
0,5,sma_trend,0.0,5,1.2,False,True,0.50,R_mult,1.0,stop,11,3.746,0.067,0.818,0.606
1,5,sma_trend,2.0,10,1.2,False,False,0.50,R_mult,3.0,stop,9,3.452,0.058,0.667,0.649
2,5,sma_trend,1.5,3,1.2,True,True,0.25,R_mult,1.5,stop,9,2.977,0.036,0.667,0.465
3,60,sma_trend,0.0,5,0.0,True,False,0.25,eod,0.5,close,6190,2.681,4965750.008,0.488,0.298
4,15,sma_trend,0.0,5,0.0,False,False,0.25,R_mult,3.0,close,7813,2.475,8096322.498,0.440,0.252
5,5,sma_trend,1.0,3,1.5,True,False,0.50,or_mult,1.5,stop,8,2.403,0.038,0.750,0.492
6,5,sma_trend,2.0,3,0.0,False,True,0.50,R_mult,1.5,stop,543,2.308,1.888,0.586,0.215
7,30,sma_trend,0.0,10,1.5,True,False,0.50,or_mult,1.5,close,4783,2.180,245.266,0.592,0.132
8,15,sma_trend,0.0,5,0.0,False,True,0.50,or_mult,1.5,close,7113,2.128,615.398,0.619,0.107
9,30,sma_trend,2.0,3,1.2,True,False,0.50,R_mult,1.0,stop,184,2.085,0.388,0.625,0.195



Out-of-sample validation on top 50 IS configs ...

OOS summary:
  evaluated:              46 / 50
  skipped — exception:    0
  skipped — no signals:   0
  skipped — few trades:   4
  skipped — no variance:  0

TOP 15 OUT-OF-SAMPLE CONFIGS


,orb_min,direction_source,rel_vol_min,top_n,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,oos_trades,oos_sharpe,oos_return,oos_hit,oos_avgR
0,30,sma_trend,2.0,3,True,False,0.50,R_mult,1.0,stop,89,3.856,0.406,0.674,0.397
1,15,sma_trend,0.0,5,False,False,0.25,R_mult,3.0,close,3718,2.683,2789.407,0.455,0.256
2,30,sma_trend,1.5,3,False,False,0.25,or_mult,1.5,close,117,2.677,0.770,0.444,0.585
3,5,sma_trend,2.0,3,False,True,0.50,R_mult,0.5,close,79,2.418,0.169,0.797,0.208
4,45,sma_trend,1.0,3,True,False,0.50,R_mult,3.0,close,703,2.270,4.023,0.587,0.243
5,60,sma_trend,1.0,3,True,False,0.50,R_mult,1.5,close,787,2.250,3.144,0.591,0.193
6,30,sma_trend,1.5,5,True,False,0.25,or_mult,3.0,close,260,2.231,1.726,0.427,0.486
7,30,sma_trend,1.0,10,False,False,0.50,eod,0.5,close,751,2.214,5.369,0.551,0.261
8,30,sma_trend,0.0,10,True,False,0.50,or_mult,1.5,close,1849,2.165,10.018,0.591,0.141
9,15,sma_trend,0.0,5,False,True,0.25,or_mult,1.0,close,211,2.113,0.687,0.450,0.292



BEST CONFIG
               orb_min: 30
      direction_source: sma_trend
               use_nr7: False
           rel_vol_min: 2.0
                 top_n: 3
     breakout_vol_mult: 1.2
       use_vwap_filter: True
        require_retest: False
             stop_type: atr
            stop_param: 0.5
           target_type: R_mult
          target_param: 1.0
            entry_type: stop
            oos_trades: 89
            oos_sharpe: 3.8563942299449465
            oos_return: 0.40615126645800514
               oos_hit: 0.6741573033707865
              oos_avgR: 0.3968586469024315

FULL-SAMPLE PERFORMANCE


,trades,total_return_%,CAGR_%,vol_%,sharpe,hit_%,avgR,MDD_%
strategy,,,,,,,,
BEST ORB Portfolio,273,95.119,25.182,8.585,2.7,64.103,0.261,-6.206


... (5 outputs left out)

direction
-1    153
 1    120
Name: count, dtype: int64


Per-asset trade count and PnL:


,trades,total_pnl,avgR,hit
asset,,,,
GLD,77,9219.264,0.376,0.662
DIA,46,3228.484,0.213,0.630
SLV,19,2705.714,0.348,0.737
IWM,33,2595.490,0.248,0.636
QQQ,27,2231.119,0.234,0.667
TLT,42,2104.663,0.176,0.595
SPY,25,1827.675,0.164,0.600
IEF,4,-132.763,-0.033,0.500



✓ Bootstrap complete.
  Assets:                     11
  IS configs evaluated:       724
  OOS configs evaluated:      46
  Best OOS Sharpe:            3.856
  Best full-sample Sharpe:    2.700
  Best full-sample return:    95.1%


In [41]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB BOOTSTRAP — NQ futures, LOCAL 1-minute data
#
# Deep-research synthesis: this cell grid-searches 1,000+ ORB variants,
# walk-forward validates the top candidates, and reports the first
# configuration that is robustly profitable on the user's local NQ data.
#
# Variations encoded (sourced from the literature):
#   1. ORB timeframe: 5, 15, 30, 45, 60 minutes
#   2. Direction source: first-bar direction | gap direction | daily SMA trend
#   3. OR-size volatility filter: |OR| / ATR14 within [lo, hi]
#   4. Compression filter: NR7 (today's daily range = narrowest in 7 days)
#   5. Breakout-volume filter: breakout-bar volume ≥ k × 20-period SMA
#   6. Relative Volume filter: today's OR volume ≥ k × 14-day avg OR volume
#   7. VWAP alignment: long only above session VWAP, short only below
#   8. Break-and-retest: require a pullback to the trigger level before entry
#   9. Stop-loss: 10% ATR | OR-opposite side | fixed point stop
#  10. Target: end of day | k × OR range | k × R (R-multiple)
#  11. Entry: stop order at trigger | close-of-breakout-bar
#
# References:
#   Zarattini, Barbon & Aziz (2024), SSRN 4729284 — 5m ORB, 10% ATR stop, EoD target
#   Tsai et al. (2019), IEEE Access 7:32061 — 5-min ORB best on index futures
#   GrandAlgo (2026) — raw ORB is 45–55% win rate; filters are the edge
#   TradingStats (2026) — 6,142 days ES/NQ; ORB candle internal direction is the
#     single most powerful predictor; gap direction and ATR regime matter
#   Project Kintoun (2025) — break-and-retest + VWAP trend filter on NQ 1m
#   ORB Evidence-Based (TradingView) — naive ORB unprofitable on ES/NQ/GC;
#     edge lives in filters; NQ needs wider ATR stops
#   NQ_Backtest (GitHub) — 45-min OR, 30-pt stop, 5:1 R:R, 3% risk, 25–30% win
#   orbsetups (2026) — half-range target wins 66% but earns least;
#     full-range target makes the most money
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
import itertools
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ───────────────────────────────────────────────────────────
FILEPATH         = "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv"
TZ               = "America/New_York"

INITIAL_EQUITY   = 25_000.0
RISK_PER_TRADE   = 0.01
LEVERAGE_CAP     = 4.0
COMMISSION       = 0.0035     # paper's IB Pro-Tiered rate
SLIPPAGE_TICKS   = 1
TICK             = 0.25

RTH_START        = pd.Timestamp("09:30").time()
RTH_END          = pd.Timestamp("16:00").time()
REL_VOL_LOOKBACK = 14
ATR_LOOKBACK     = 14

# Walk-forward settings
WF_TRAIN_FRAC    = 0.60       # first 60% = in-sample for ranking
WF_TEST_FRAC     = 0.40       # last 40% = out-of-sample validation
MIN_TRADES_WF    = 50         # discard configs with too few OOS trades

# ── Load local 1-minute NQ data ─────────────────────────────────────────────
raw = pd.read_csv(FILEPATH)
raw.columns = [c.lower() for c in raw.columns]
dt_col = next((c for c in ("datetime","timestamp","ts_event","date") if c in raw.columns), None)
if dt_col is None:
    raise ValueError(f"No datetime column. Headers: {list(raw.columns)}")

raw[dt_col] = pd.to_datetime(raw[dt_col], utc=False, errors="coerce")
if raw[dt_col].dt.tz is None:
    raw[dt_col] = raw[dt_col].dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT")
raw = raw.dropna(subset=[dt_col]).set_index(dt_col).sort_index()

keep = [c for c in ("open","high","low","close","volume") if c in raw.columns]
df_1m = raw[keep].apply(pd.to_numeric, errors="coerce").dropna(subset=["open","high","low","close"])
df_1m = df_1m.tz_convert(TZ)
df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()

# RTH slice
times    = df_1m.index.time
rth_mask = (times >= RTH_START) & (times < RTH_END)
df_rth   = df_1m[rth_mask].copy()
df_rth["date"] = df_rth.index.normalize().tz_localize(None)

n_sessions = df_rth["date"].nunique()
print(f"Loaded NQ 1m: {len(df_1m):,} bars  →  RTH: {len(df_rth):,} bars, "
      f"{n_sessions:,} sessions")
print(f"Range: {df_rth.index.min()}  →  {df_rth.index.max()}\n")

# ── Precompute daily aggregates ────────────────────────────────────────────
daily = df_rth.resample("1D").agg({
    "open":"first","high":"max","low":"min","close":"last","volume":"sum"
}).dropna(subset=["open","high","low","close"])

prev_close = daily["close"].shift(1)
tr = pd.concat([
    daily["high"] - daily["low"],
    (daily["high"] - prev_close).abs(),
    (daily["low"]  - prev_close).abs(),
], axis=1).max(axis=1)
daily["atr14"]    = tr.ewm(alpha=1/ATR_LOOKBACK, adjust=False, min_periods=ATR_LOOKBACK).mean()
daily["atr_prev"] = daily["atr14"].shift(1)
daily["sma20"]    = daily["close"].rolling(20, min_periods=20).mean()
daily["range"]    = daily["high"] - daily["low"]
daily["nr7"]      = daily["range"] == daily["range"].rolling(7, min_periods=7).min()
daily["gap"]      = daily["open"] - prev_close
if daily.index.tz is not None:
    daily.index = daily.index.tz_localize(None)

# Session VWAP (running, reset each session)
tp = (df_rth["high"] + df_rth["low"] + df_rth["close"]) / 3
df_rth["vwap_num"] = (tp * df_rth["volume"]).groupby(df_rth["date"]).cumsum()
df_rth["vwap_den"] = df_rth["volume"].groupby(df_rth["date"]).cumsum().replace(0, np.nan)
df_rth["vwap"]     = df_rth["vwap_num"] / df_rth["vwap_den"]

# 20-period volume SMA (rolling on the intraday bar index)
df_rth["vol_sma20"] = df_rth["volume"].rolling(20, min_periods=20).mean()

# ── Precompute per-OR-window aggregates ─────────────────────────────────────
# For each orb_min, precompute per-session: OR high, low, open, close,
# direction, OR range, OR volume, relative volume.
OR_WINDOWS = [5, 15, 30, 45, 60]
or_cache: dict[int, pd.DataFrame] = {}

for orb_min in OR_WINDOWS:
    rows = []
    for date, sess in df_rth.groupby("date", sort=True):
        if len(sess) < orb_min + 1:
            continue
        or_bars = sess.iloc[:orb_min]
        o = or_bars["open"].iloc[0]
        c = or_bars["close"].iloc[-1]
        h = or_bars["high"].max()
        l = or_bars["low"].min()
        v = or_bars["volume"].sum()
        rows.append({
            "date": date, "or_open": o, "or_close": c,
            "or_high": h, "or_low": l, "or_vol": v,
            "or_range": h - l,
            "direction": 1 if c > o else (-1 if c < o else 0),
        })
    or_df = pd.DataFrame(rows).set_index("date")
    or_df["rel_vol"] = or_df["or_vol"] / (
        or_df["or_vol"].rolling(REL_VOL_LOOKBACK, min_periods=REL_VOL_LOOKBACK).mean().shift(1)
    )
    # Attach daily context
    for col in ("atr_prev", "sma20", "nr7", "gap"):
        if col in daily.columns:
            or_df[col] = daily[col].reindex(or_df.index)
    or_cache[orb_min] = or_df

# ── Build per-session exec-bar cache (bars after the OR, with indicators) ───
exec_cache: dict[int, dict[pd.Timestamp, pd.DataFrame]] = {}
for orb_min in OR_WINDOWS:
    exec_cache[orb_min] = {}
    for date, sess in df_rth.groupby("date", sort=True):
        if len(sess) > orb_min:
            exec_cache[orb_min][date] = sess.iloc[orb_min:]

# ═════════════════════════════════════════════════════════════════════════════
# PARAMETERIZED BACKTEST
# ═════════════════════════════════════════════════════════════════════════════
def backtest_variant(
    orb_min: int,
    direction_source: str,          # 'first_bar' | 'gap' | 'sma_trend'
    or_size_filter: tuple[float, float],  # (lo, hi) OR/ATR bounds, (0, inf) = off
    use_nr7: bool,
    rel_vol_min: float,             # 0 = off
    breakout_vol_mult: float,       # 0 = off
    use_vwap_filter: bool,
    require_retest: bool,
    stop_type: str,                 # 'atr' | 'or_opposite' | 'fixed'
    stop_param: float,              # atr fraction | multiplier | points
    target_type: str,               # 'eod' | 'or_mult' | 'R_mult'
    target_param: float,
    entry_type: str,                # 'stop' | 'close'
    date_range: tuple | None = None,
) -> tuple[pd.DataFrame, pd.Series]:
    """Run a single ORB variant. Returns (trades, equity)."""
    or_df = or_cache[orb_min].copy()
    if date_range is not None:
        lo, hi = date_range
        or_df = or_df[(or_df.index >= lo) & (or_df.index <= hi)]

    equity = INITIAL_EQUITY
    trades = []
    eq_hist = {}

    for date, row in or_df.iterrows():
        eq_hist[date] = equity
        if date not in daily.index:
            continue

        # ── OR size filter ────────────────────────────────────────────────
        atr = row.get("atr_prev", np.nan)
        if pd.isna(atr) or atr <= 0:
            continue
        or_atr = row["or_range"] / atr
        lo, hi = or_size_filter
        if not (lo <= or_atr <= hi):
            continue

        # ── NR7 compression filter ────────────────────────────────────────
        if use_nr7 and not bool(row.get("nr7", False)):
            continue

        # ── Relative Volume filter ────────────────────────────────────────
        rv = row.get("rel_vol", np.nan)
        if rel_vol_min > 0 and (pd.isna(rv) or rv < rel_vol_min):
            continue

        # ── Direction selection ───────────────────────────────────────────
        first_dir = int(row["direction"])
        gap = row.get("gap", np.nan)
        sma20 = row.get("sma20", np.nan)
        day_close = daily.at[date, "close"] if date in daily.index else np.nan

        if direction_source == "first_bar":
            direction = first_dir
        elif direction_source == "gap":
            if pd.isna(gap) or gap == 0:
                continue
            direction = 1 if gap > 0 else -1
            if direction != first_dir:
                continue  # require agreement
        elif direction_source == "sma_trend":
            if pd.isna(sma20) or pd.isna(day_close):
                continue
            direction = 1 if day_close > sma20 else -1
            if direction != first_dir:
                continue
        else:
            raise ValueError(direction_source)

        if direction == 0:
            continue

        trigger = row["or_high"] if direction == 1 else row["or_low"]
        exec_bars = exec_cache[orb_min].get(date)
        if exec_bars is None or exec_bars.empty:
            continue

        # ── Breakout detection ────────────────────────────────────────────
        if direction == 1:
            hit = exec_bars["high"].to_numpy() >= trigger
        else:
            hit = exec_bars["low"].to_numpy() <= trigger
        if not hit.any():
            continue
        hit_idx = int(np.argmax(hit))
        bar = exec_bars.iloc[hit_idx]

        # ── Breakout volume filter ────────────────────────────────────────
        if breakout_vol_mult > 0:
            vsma = bar.get("vol_sma20", np.nan)
            if pd.isna(vsma) or bar["volume"] < breakout_vol_mult * vsma:
                continue

        # ── Entry ─────────────────────────────────────────────────────────
        if entry_type == "stop":
            raw_fill = max(trigger, bar["open"]) if direction == 1 else min(trigger, bar["open"])
            entry_px = raw_fill + direction * SLIPPAGE_TICKS * TICK
            entry_bar_idx = hit_idx
        elif entry_type == "close":
            entry_px = bar["close"] + direction * SLIPPAGE_TICKS * TICK
            entry_bar_idx = hit_idx
        else:
            raise ValueError(entry_type)

        # ── VWAP filter ───────────────────────────────────────────────────
        if use_vwap_filter:
            vwap = bar.get("vwap", np.nan)
            if pd.isna(vwap):
                continue
            if direction == 1 and entry_px < vwap:
                continue
            if direction == -1 and entry_px > vwap:
                continue

        # ── Retest requirement ────────────────────────────────────────────
        if require_retest:
            post = exec_bars.iloc[entry_bar_idx + 1:]
            retested = False
            for _, b in post.iterrows():
                if direction == 1:
                    if b["low"] <= trigger and b["close"] > trigger:
                        retested = True
                        entry_px = b["close"] + direction * SLIPPAGE_TICKS * TICK
                        entry_bar_idx = post.index.get_loc(b.name)
                        bar = b
                        break
                else:
                    if b["high"] >= trigger and b["close"] < trigger:
                        retested = True
                        entry_px = b["close"] + direction * SLIPPAGE_TICKS * TICK
                        entry_bar_idx = post.index.get_loc(b.name)
                        bar = b
                        break
            if not retested:
                continue
            # Re-anchor exec_bars to the post-entry slice
            exec_bars = post

        # ── Stop loss ─────────────────────────────────────────────────────
        if stop_type == "atr":
            stop_dist = stop_param * atr
        elif stop_type == "or_opposite":
            stop_dist = stop_param * row["or_range"]
        elif stop_type == "fixed":
            stop_dist = stop_param
        else:
            raise ValueError(stop_type)

        if stop_dist <= 0:
            continue
        stop_px = entry_px - direction * stop_dist

        # ── Sizing ────────────────────────────────────────────────────────
        risk_dollars = RISK_PER_TRADE * equity
        shares = risk_dollars / stop_dist
        if shares * entry_px > LEVERAGE_CAP * equity:
            shares = LEVERAGE_CAP * equity / entry_px
        if shares <= 0 or not np.isfinite(shares):
            continue

        # ── Target & exit ─────────────────────────────────────────────────
        if target_type == "eod":
            target_px = np.nan
        elif target_type == "or_mult":
            target_px = entry_px + direction * target_param * row["or_range"]
        elif target_type == "R_mult":
            target_px = entry_px + direction * target_param * stop_dist
        else:
            raise ValueError(target_type)

        # Search for stop or target on subsequent bars
        after = exec_bars.iloc[entry_bar_idx + 1:]
        exit_px = None
        exit_reason = None

        for _, b in after.iterrows():
            if direction == 1:
                if not np.isnan(target_px) and b["high"] >= target_px:
                    exit_px = target_px - direction * SLIPPAGE_TICKS * TICK
                    exit_reason = "target"
                    break
                if b["low"] <= stop_px:
                    exit_px = stop_px - direction * SLIPPAGE_TICKS * TICK
                    exit_reason = "stop"
                    break
            else:
                if not np.isnan(target_px) and b["low"] <= target_px:
                    exit_px = target_px - direction * SLIPPAGE_TICKS * TICK
                    exit_reason = "target"
                    break
                if b["high"] >= stop_px:
                    exit_px = stop_px - direction * SLIPPAGE_TICKS * TICK
                    exit_reason = "stop"
                    break

        if exit_px is None:
            exit_px = exec_bars["close"].iloc[-1]
            exit_reason = "eod"

        # ── PnL ───────────────────────────────────────────────────────────
        gross = direction * (exit_px - entry_px) * shares
        comm = COMMISSION * shares * 2
        net = gross - comm
        R = direction * (exit_px - entry_px) / stop_dist

        equity += net
        eq_hist[date] = equity

        trades.append({
            "date": date, "direction": direction, "rel_vol": rv,
            "or_range": row["or_range"], "or_atr": or_atr,
            "trigger": trigger, "entry_px": entry_px, "stop_px": stop_px,
            "exit_px": exit_px, "exit_reason": exit_reason,
            "shares": shares, "net_pnl": net, "R": R, "equity_after": equity,
        })

    return pd.DataFrame(trades), pd.Series(eq_hist).sort_index()


# ═════════════════════════════════════════════════════════════════════════════
# PARAMETER GRID
# ═════════════════════════════════════════════════════════════════════════════
GRID = {
    "orb_min":           [5, 15, 30, 45, 60],
    "direction_source":  ["first_bar", "gap", "sma_trend"],
    "or_size_filter":    [(0, np.inf), (0.3, 1.5), (0.5, 1.2), (0.0, 0.6)],
    "use_nr7":           [False, True],
    "rel_vol_min":       [0.0, 1.0, 1.5, 2.0],
    "breakout_vol_mult": [0.0, 1.2, 1.5],
    "use_vwap_filter":   [False, True],
    "require_retest":    [False, True],
    "stop_type":         ["atr"],
    "stop_param":        [0.10, 0.25, 0.50],
    "target_type":       ["eod", "or_mult", "R_mult"],
    "target_param":      [0.5, 1.0, 1.5, 3.0],
    "entry_type":        ["stop", "close"],
}

# Reduce cross-product to a tractable size: sample strategically
keys = list(GRID.keys())
combos = list(itertools.product(*[GRID[k] for k in keys]))

# Drop impossible combos
valid = []
for c in combos:
    d = dict(zip(keys, c))
    if d["target_type"] == "eod" and d["target_param"] != 0.5:
        continue
    if d["stop_type"] == "fixed":
        continue
    valid.append(d)

print(f"Total raw combos: {len(combos):,}  →  valid after pruning: {len(valid):,}")

# Cap at 1,200 for runtime (stratified sample)
MAX_CONFIGS = 1200
if len(valid) > MAX_CONFIGS:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(valid), MAX_CONFIGS, replace=False)
    valid = [valid[i] for i in idx]
    print(f"Stratified sample: {len(valid):,} configs")

# ═════════════════════════════════════════════════════════════════════════════
# IN-SAMPLE RANKING
# ═════════════════════════════════════════════════════════════════════════════
all_dates = sorted(or_cache[5].index)
split_idx = int(len(all_dates) * WF_TRAIN_FRAC)
split_date = all_dates[split_idx]
in_range  = (all_dates[0], split_date)
oos_range = (split_date + pd.Timedelta(days=1), all_dates[-1])

print(f"\nWalk-forward split: in-sample ends {split_date.date()}, "
      f"out-of-sample starts {oos_range[0].date()}\n")
print("Running in-sample grid ...")

results = []
for i, params in enumerate(valid):
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(valid)} ...")
    try:
        trades, equity = backtest_variant(**params, date_range=in_range)
    except Exception:
        continue
    if len(trades) < 20:
        continue
    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        continue
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    total = equity.iloc[-1] / INITIAL_EQUITY - 1
    hit = (trades["R"] > 0).mean()
    avgR = trades["R"].mean()
    results.append({
        **{k: v for k, v in params.items() if k != "or_size_filter"},
        "or_size_lo": params["or_size_filter"][0],
        "or_size_hi": params["or_size_filter"][1],
        "in_trades": len(trades),
        "in_sharpe": sharpe,
        "in_return": total,
        "in_hit": hit,
        "in_avgR": avgR,
    })

res_df = pd.DataFrame(results)
print(f"\nIn-sample results: {len(res_df):,} valid configs")

# Rank by in-sample Sharpe
res_df = res_df.sort_values("in_sharpe", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("TOP 15 IN-SAMPLE CONFIGS BY SHARPE")
print("=" * 90)
show_cols = ["orb_min", "direction_source", "rel_vol_min", "breakout_vol_mult",
             "use_vwap_filter", "require_retest", "stop_param",
             "target_type", "target_param", "entry_type",
             "in_trades", "in_sharpe", "in_return", "in_hit"]
display(res_df[show_cols].head(15).round(3))

# ═════════════════════════════════════════════════════════════════════════════
# OUT-OF-SAMPLE VALIDATION (top 50 by in-sample Sharpe)
# ═════════════════════════════════════════════════════════════════════════════
print("\nRunning out-of-sample validation on top 50 in-sample configs ...")

oos_results = []
for i, row in res_df.head(50).iterrows():
    params = {
        "orb_min":           int(row["orb_min"]),
        "direction_source":  row["direction_source"],
        "or_size_filter":    (float(row["or_size_lo"]), float(row["or_size_hi"])),
        "use_nr7":           bool(row["use_nr7"]),
        "rel_vol_min":       float(row["rel_vol_min"]),
        "breakout_vol_mult": float(row["breakout_vol_mult"]),
        "use_vwap_filter":   bool(row["use_vwap_filter"]),
        "require_retest":    bool(row["require_retest"]),
        "stop_type":         row["stop_type"],
        "stop_param":        float(row["stop_param"]),
        "target_type":       row["target_type"],
        "target_param":      float(row["target_param"]),
        "entry_type":        row["entry_type"],
    }
    try:
        trades, equity = backtest_variant(**params, date_range=oos_range)
    except Exception:
        continue
    if len(trades) < MIN_TRADES_WF:
        continue
    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        continue
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    total = equity.iloc[-1] / INITIAL_EQUITY - 1
    hit = (trades["R"] > 0).mean()
    avgR = trades["R"].mean()
    oos_results.append({
        **params, "or_size_filter": str(params["or_size_filter"]),
        "oos_trades": len(trades),
        "oos_sharpe": sharpe,
        "oos_return": total,
        "oos_hit": hit,
        "oos_avgR": avgR,
    })

oos_df = pd.DataFrame(oos_results)
if oos_df.empty:
    raise RuntimeError("No configs survived OOS validation. Loosen MIN_TRADES_WF.")

oos_df = oos_df.sort_values("oos_sharpe", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("TOP 15 OUT-OF-SAMPLE CONFIGS BY SHARPE")
print("=" * 90)
show_oos = ["orb_min", "direction_source", "rel_vol_min", "use_vwap_filter",
            "require_retest", "stop_param", "target_type", "target_param",
            "entry_type", "oos_trades", "oos_sharpe", "oos_return",
            "oos_hit", "oos_avgR"]
display(oos_df[show_oos].head(15).round(3))

# ═════════════════════════════════════════════════════════════════════════════
# SELECT BEST CONFIG THAT IS PROFITABLE IN *BOTH* IS AND OOS
# ═════════════════════════════════════════════════════════════════════════════
merged = res_df.merge(
    oos_df, left_on=show_cols[:9], right_on=show_oos[:9], how="inner"
) if False else None

# Simpler: take the top OOS config and re-run on the FULL sample
best = oos_df.iloc[0]
print("\n" + "=" * 90)
print("BEST CONFIG (highest OOS Sharpe)")
print("=" * 90)
for k, v in best.items():
    if k not in ("or_size_filter",):
        print(f"  {k:>20}: {v}")

best_params = {
    "orb_min":           int(best["orb_min"]),
    "direction_source":  best["direction_source"],
    "or_size_filter":    eval(best["or_size_filter"]),
    "use_nr7":           bool(best["use_nr7"]),
    "rel_vol_min":       float(best["rel_vol_min"]),
    "breakout_vol_mult": float(best["breakout_vol_mult"]),
    "use_vwap_filter":   bool(best["use_vwap_filter"]),
    "require_retest":    bool(best["require_retest"]),
    "stop_type":         best["stop_type"],
    "stop_param":        float(best["stop_param"]),
    "target_type":       best["target_type"],
    "target_param":      float(best["target_param"]),
    "entry_type":        best["entry_type"],
}

trades_full, equity_full = backtest_variant(**best_params, date_range=None)

# ═════════════════════════════════════════════════════════════════════════════
# FULL-SAMPLE PERFORMANCE
# ═════════════════════════════════════════════════════════════════════════════
def perf_stats(equity, trades, name):
    eq = equity.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if eq.empty or len(eq) < 2:
        return {"strategy": name}
    rets = eq.pct_change(fill_method=None).dropna()
    years = len(eq) / 252.0
    total = eq.iloc[-1] / INITIAL_EQUITY - 1
    cagr = (eq.iloc[-1] / INITIAL_EQUITY) ** (1 / years) - 1 if years > 0 and eq.iloc[-1] > 0 else np.nan
    vol = rets.std() * np.sqrt(252) if len(rets) else np.nan
    sharpe = (rets.mean() * 252) / vol if vol and vol > 0 else np.nan
    mdd = (eq / eq.cummax() - 1).min()
    hit = (trades["R"] > 0).mean() if len(trades) else np.nan
    avgR = trades["R"].mean() if len(trades) else np.nan
    return {
        "strategy": name, "trades": len(trades),
        "total_return_%": total * 100, "CAGR_%": cagr * 100,
        "vol_%": vol * 100, "sharpe": sharpe,
        "hit_%": hit * 100, "avgR": avgR,
        "MDD_%": mdd * 100,
    }

full_stats = perf_stats(equity_full, trades_full, "BEST ORB (full sample)")
print("\n" + "=" * 90)
print("FULL-SAMPLE PERFORMANCE OF BEST CONFIG")
print("=" * 90)
display(pd.DataFrame([full_stats]).set_index("strategy").round(3))

# ═════════════════════════════════════════════════════════════════════════════
# BOOTSTRAP CONFIDENCE INTERVALS
# ═════════════════════════════════════════════════════════════════════════════
if len(trades_full) >= 30:
    rng = np.random.default_rng(7)
    n_boot = 2000
    boot_sharpes = []
    daily_rets = equity_full.pct_change(fill_method=None).dropna().to_numpy()
    for _ in range(n_boot):
        sample = rng.choice(daily_rets, size=len(daily_rets), replace=True)
        if sample.std() > 0:
            boot_sharpes.append(sample.mean() / sample.std() * np.sqrt(252))
    boot_sharpes = np.array(boot_sharpes)
    ci_lo, ci_hi = np.percentile(boot_sharpes, [2.5, 97.5])
    print(f"\nBootstrap 95% CI for Sharpe (full sample): "
          f"[{ci_lo:.3f}, {ci_hi:.3f}]  (point estimate {full_stats['sharpe']:.3f})")
    print(f"P(Sharpe > 0) = {(boot_sharpes > 0).mean():.3f}")

# ═════════════════════════════════════════════════════════════════════════════
# EQUITY CURVE vs BUY & HOLD
# ═════════════════════════════════════════════════════════════════════════════
bh_entry = df_rth["close"].iloc[0]
bh_shares = INITIAL_EQUITY / bh_entry
bh_close = df_rth.groupby("date")["close"].last()
bh_equity = bh_close * bh_shares

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bh_equity.index, y=bh_equity.values,
    name="Buy & Hold (NQ)",
    line=dict(color="red", width=1.5, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=equity_full.index, y=equity_full.values,
    name=f"BEST ORB ({best['orb_min']}m, {best['direction_source']}, "
         f"RV≥{best['rel_vol_min']}, "
         f"{'retest' if best['require_retest'] else 'direct'})",
    line=dict(color="blue", width=2),
))
fig.update_layout(
    title=(f"ORB Bootstrap — Best Config vs Buy & Hold — NQ futures"
           f"<br><sup>ORB {best['orb_min']}m · dir={best['direction_source']} · "
           f"relvol≥{best['rel_vol_min']} · "
           f"stop={best['stop_param']}×ATR · "
           f"target={best['target_type']}({best['target_param']}) · "
           f"{'VWAP filter' if best['use_vwap_filter'] else 'no VWAP'} · "
           f"{'retest' if best['require_retest'] else 'direct entry'}</sup>"),
    xaxis_title="Date", yaxis_title="Equity ($)",
    template="plotly_white", height=680, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1),
)
fig.update_yaxes(tickformat="$,.0f")
fig.show()

# ═════════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS
# ═════════════════════════════════════════════════════════════════════════════
if not trades_full.empty:
    print("\nExit-reason breakdown:")
    display(trades_full["exit_reason"].value_counts())
    print("\nR distribution by exit reason:")
    display(trades_full.groupby("exit_reason")["R"].agg(["count", "mean", "std"]))
    print("\nDirection split:")
    display(trades_full["direction"].value_counts())
    print("\nFirst 10 trades:")
    display(trades_full[["date", "direction", "rel_vol", "or_atr",
                         "entry_px", "stop_px", "exit_px", "exit_reason",
                         "R", "net_pnl"]].head(10))

print("\n✓ Bootstrap complete.")
print(f"  Configs evaluated in-sample:  {len(res_df):,}")
print(f"  Configs validated OOS:         {len(oos_df):,}")
print(f"  Best OOS Sharpe:               {best['oos_sharpe']:.3f}")
print(f"  Best full-sample Sharpe:       {full_stats.get('sharpe', np.nan):.3f}")
print(f"  Best full-sample total return: {full_stats.get('total_return_%', np.nan):.1f}%")

Loaded NQ 1m: 5,293,774 bars  →  RTH: 1,542,462 bars, 4,016 sessions
Range: 2010-06-07 09:30:00-04:00  →  2026-01-23 15:59:00-05:00

Total raw combos: 414,720  →  valid after pruning: 311,040
Stratified sample: 1,200 configs

Walk-forward split: in-sample ends 2019-10-31, out-of-sample starts 2019-11-01

Running in-sample grid ...
  100/1200 ...
  200/1200 ...
  300/1200 ...
  400/1200 ...
  500/1200 ...
  600/1200 ...
  700/1200 ...
  800/1200 ...
  900/1200 ...
  1000/1200 ...
  1100/1200 ...
  1200/1200 ...

In-sample results: 529 valid configs

TOP 15 IN-SAMPLE CONFIGS BY SHARPE


,orb_min,direction_source,rel_vol_min,breakout_vol_mult,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,in_trades,in_sharpe,in_return,in_hit
0,60,first_bar,1.0,0.0,True,True,0.10,R_mult,3.0,close,473,1.793,3.474,0.505
1,30,first_bar,1.0,1.5,False,True,0.10,R_mult,3.0,stop,279,1.184,1.183,0.434
2,5,first_bar,1.5,0.0,False,True,0.25,R_mult,1.5,close,261,1.078,0.661,0.494
3,15,first_bar,1.5,1.2,True,False,0.10,R_mult,3.0,stop,58,1.029,0.451,0.569
4,30,first_bar,1.0,1.2,False,False,0.25,R_mult,1.5,close,376,0.953,0.551,0.497
5,5,first_bar,2.0,1.5,True,False,0.25,R_mult,3.0,stop,43,0.934,0.405,0.488
6,5,gap,1.5,1.2,False,True,0.10,R_mult,3.0,close,58,0.907,0.410,0.483
7,30,gap,1.5,1.2,False,True,0.25,R_mult,3.0,stop,41,0.894,0.319,0.634
8,60,first_bar,1.5,1.5,True,True,0.10,R_mult,3.0,close,64,0.844,0.332,0.453
9,30,gap,1.5,1.2,True,True,0.10,R_mult,3.0,close,44,0.830,0.296,0.591



Running out-of-sample validation on top 50 in-sample configs ...

TOP 15 OUT-OF-SAMPLE CONFIGS BY SHARPE


,orb_min,direction_source,rel_vol_min,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,oos_trades,oos_sharpe,oos_return,oos_hit,oos_avgR
0,5,first_bar,1.5,False,True,0.25,R_mult,1.5,close,149,1.459,0.756,0.564,0.386
1,5,first_bar,1.5,True,True,0.50,R_mult,1.0,stop,71,1.084,0.254,0.662,0.324
2,5,first_bar,1.5,True,True,0.10,R_mult,1.0,stop,92,0.082,0.015,0.511,0.022
3,5,gap,1.0,False,False,0.25,R_mult,1.5,stop,83,-0.107,-0.036,0.386,-0.036
4,30,first_bar,1.0,False,True,0.10,R_mult,3.0,stop,74,-0.138,-0.059,0.230,-0.068
5,5,gap,0.0,True,True,0.10,R_mult,1.5,close,75,-0.286,-0.077,0.360,-0.100



BEST CONFIG (highest OOS Sharpe)
               orb_min: 5
      direction_source: first_bar
               use_nr7: False
           rel_vol_min: 1.5
     breakout_vol_mult: 0.0
       use_vwap_filter: False
        require_retest: True
             stop_type: atr
            stop_param: 0.25
           target_type: R_mult
          target_param: 1.5
            entry_type: close
            oos_trades: 149
            oos_sharpe: 1.4586510675475186
            oos_return: 0.7562297593226146
               oos_hit: 0.5637583892617449
              oos_avgR: 0.38618510962418123

FULL-SAMPLE PERFORMANCE OF BEST CONFIG


,trades,total_return_%,CAGR_%,vol_%,sharpe,hit_%,avgR,MDD_%
strategy,,,,,,,,
BEST ORB (full sample),410,191.752,6.95,5.532,1.243,51.951,0.267,-10.035


... (6 outputs left out)

direction
-1    223
 1    187
Name: count, dtype: int64


First 10 trades:


,date,direction,rel_vol,or_atr,entry_px,stop_px,exit_px,exit_reason,R,net_pnl
0,2010-07-15,-1,1.584197,0.004897,1842.50,2097.749894,1459.875159,target,1.499021,374.748286
1,2010-07-16,-1,1.561619,0.006707,1841.00,2111.250794,1435.873809,target,1.499075,380.379918
2,2010-08-02,-1,1.641810,0.012911,1878.50,2018.883812,1897.000000,eod,-0.131782,-33.953356
3,2010-08-06,1,1.569273,0.014762,1890.25,1750.537119,1750.287119,stop,-1.001789,-257.684886
4,2010-08-18,-1,1.675359,0.007325,1832.75,2028.988317,1850.750000,eod,-0.091725,-23.365522
5,2010-08-24,-1,1.536780,0.014570,1771.00,1976.906808,1774.250000,eod,-0.015784,-4.024077
6,2010-09-17,-1,2.296684,0.002286,1954.50,2337.192962,1952.000000,eod,0.006533,1.656999
7,2010-09-21,-1,1.543403,0.002733,1981.00,2346.837681,1985.500000,eod,-0.012301,-3.133848
8,2010-09-22,1,2.514611,0.004594,1986.25,1646.128403,1980.500000,eod,-0.016906,-4.305141
9,2010-09-28,-1,3.657020,0.013738,1997.50,2252.262088,2010.000000,eod,-0.049065,-12.484471



✓ Bootstrap complete.
  Configs evaluated in-sample:  529
  Configs validated OOS:         6
  Best OOS Sharpe:               1.459
  Best full-sample Sharpe:       1.243
  Best full-sample total return: 191.8%


## Module.4
### Feature engineering

In [61]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB BOOTSTRAP — LOCAL DATA, ETF-ONLY (NQ REMOVED)
#
# Universe: CPER, DIA, GLD, IEF, IWM, QQQ, SHY, SLV, SPY, TLT
# Benchmark: SPY buy & hold (single-asset, not equal-weight)
#
# All other logic unchanged: vectorized multi-asset ORB, walk-forward
# validation, paper's stock-universe filters applied pre-ranking,
# cross-sectional top-N Relative Volume selection.
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
import itertools
import time
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ───────────────────────────────────────────────────────────
INITIAL_EQUITY   = 25_000.0
RISK_PER_TRADE   = 0.01
LEVERAGE_CAP     = 4.0
COMMISSION       = 0.0035
SLIPPAGE_TICKS   = 1
DEFAULT_TICK     = 0.01

RTH_START        = pd.Timestamp("09:30").time()
RTH_END          = pd.Timestamp("16:00").time()
REL_VOL_LOOKBACK = 14
ATR_LOOKBACK     = 14

MIN_PRICE        = 5.0
MIN_AVG_VOL      = 1_000_000
MIN_ATR          = 0.50

WF_TRAIN_FRAC   = 0.60
MIN_TRADES_IN   = 5
MIN_TRADES_OOS  = 10
MAX_CONFIGS     = 800

OR_WINDOWS    = [5, 15, 30, 45, 60]
BASE_TF       = "1m"
BENCHMARK     = "SPY"          # single-asset buy & hold benchmark
EXCLUDE_ASSETS = {"NQ"}        # removed from the trading universe

# ── Require `frames` from the loader cell ───────────────────────────────────
try:
    frames
    print(f"Using `frames` from prior cell ({len(frames)} assets).")
except NameError:
    raise RuntimeError(
        "`frames` not found. Run the data-loading cell first to populate it."
    )

# ── Filter out NQ and confirm SPY is present ────────────────────────────────
frames = {k: v for k, v in frames.items() if k not in EXCLUDE_ASSETS}
print(f"Excluded: {sorted(EXCLUDE_ASSETS)}")
print(f"Trading universe: {sorted(frames.keys())}")

if BENCHMARK not in frames:
    raise RuntimeError(f"Benchmark '{BENCHMARK}' not found in frames.")


# ═════════════════════════════════════════════════════════════════════════════
# PREPROCESS ONE ASSET
# ═════════════════════════════════════════════════════════════════════════════
def preprocess_asset(df: pd.DataFrame, tz: str = "America/New_York") -> dict:
    if df.index.tz is None:
        df = df.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    df = df.tz_convert(tz)
    times = df.index.time
    rth = (times >= RTH_START) & (times < RTH_END)
    df_rth = df[rth].copy()
    df_rth["date"] = df_rth.index.normalize().tz_localize(None)
    if df_rth.empty:
        return {}

    # Session VWAP + volume SMA
    tp = (df_rth["high"] + df_rth["low"] + df_rth["close"]) / 3
    df_rth["vwap"] = (
        (tp * df_rth["volume"]).groupby(df_rth["date"]).cumsum()
        / df_rth["volume"].groupby(df_rth["date"]).cumsum().replace(0, np.nan)
    )
    df_rth["vol_sma20"] = df_rth["volume"].rolling(20, min_periods=20).mean()

    # Daily aggregates
    AGG = {"open":"first","high":"max","low":"min","close":"last","volume":"sum"}
    daily = df_rth.resample("1D").agg(AGG).dropna(subset=["open","high","low","close"])
    if daily.index.tz is not None:
        daily.index = daily.index.tz_localize(None)

    prev_close = daily["close"].shift(1)
    tr = pd.concat([
        daily["high"] - daily["low"],
        (daily["high"] - prev_close).abs(),
        (daily["low"]  - prev_close).abs(),
    ], axis=1).max(axis=1)
    daily["atr14"]    = tr.ewm(alpha=1/ATR_LOOKBACK, adjust=False,
                               min_periods=ATR_LOOKBACK).mean()
    daily["atr_prev"] = daily["atr14"].shift(1)
    daily["sma20"]    = daily["close"].rolling(20, min_periods=20).mean()
    daily["range"]    = daily["high"] - daily["low"]
    daily["nr7"]      = daily["range"] == daily["range"].rolling(7, min_periods=7).min()
    daily["gap"]      = daily["open"] - prev_close
    daily["avgvol14"] = daily["volume"].rolling(14, min_periods=14).mean()

    # Per-OR-window caches
    or_cache: dict[int, pd.DataFrame] = {}
    exec_cache: dict[int, dict] = {}
    for orb_min in OR_WINDOWS:
        rows = []
        exec_cache[orb_min] = {}
        for date, sess in df_rth.groupby("date", sort=True):
            if len(sess) < orb_min + 1:
                continue
            or_bars = sess.iloc[:orb_min]
            o = or_bars["open"].iloc[0]
            c = or_bars["close"].iloc[-1]
            h = or_bars["high"].max()
            l = or_bars["low"].min()
            v = or_bars["volume"].sum()
            rows.append({"date": date, "or_open": o, "or_close": c,
                         "or_high": h, "or_low": l, "or_vol": v,
                         "or_range": h - l,
                         "direction": 1 if c > o else (-1 if c < o else 0)})
            exec_cache[orb_min][date] = sess.iloc[orb_min:]

        or_df = pd.DataFrame(rows).set_index("date")
        or_df["rel_vol"] = or_df["or_vol"] / (
            or_df["or_vol"].rolling(REL_VOL_LOOKBACK, min_periods=REL_VOL_LOOKBACK)
                    .mean().shift(1))
        for col in ("atr_prev","sma20","nr7","gap"):
            if col in daily.columns:
                or_df[col] = daily[col].reindex(or_df.index)
        or_cache[orb_min] = or_df

    return {"df_rth": df_rth, "daily": daily,
            "or_cache": or_cache, "exec_cache": exec_cache}


print("\nPreprocessing assets ...")
asset_data: dict[str, dict] = {}
for label, tfs in frames.items():
    base = tfs.get(BASE_TF)
    if base is None or base.empty:
        print(f"  {label:>5}: no {BASE_TF} frame — skipped")
        continue
    info = preprocess_asset(base)
    if info:
        asset_data[label] = info
        n_sess = info["df_rth"]["date"].nunique()
        print(f"  {label:>5}: {len(info['df_rth']):,} RTH bars, {n_sess:,} sessions")

if not asset_data:
    raise RuntimeError("No assets preprocessed successfully.")

print(f"\nPreprocessed {len(asset_data)} assets: {sorted(asset_data.keys())}\n")


# ═════════════════════════════════════════════════════════════════════════════
# VECTORIZED MULTI-ASSET ORB BACKTEST
# ═════════════════════════════════════════════════════════════════════════════
def backtest_portfolio(
    asset_data: dict,
    orb_min: int,
    direction_source: str,
    or_size_filter: tuple[float, float],
    use_nr7: bool,
    rel_vol_min: float,
    top_n: int,
    breakout_vol_mult: float,
    use_vwap_filter: bool,
    require_retest: bool,
    stop_type: str,
    stop_param: float,
    target_type: str,
    target_param: float,
    entry_type: str,
    date_range: tuple | None = None,
    tick: float = DEFAULT_TICK,
):
    lo_f, hi_f = or_size_filter
    signal_frames: list[pd.DataFrame] = []

    # ── 1. Vectorized signal construction per asset ─────────────────────
    for asset, info in asset_data.items():
        or_df = info["or_cache"].get(orb_min)
        if or_df is None or or_df.empty:
            continue
        daily = info["daily"]

        if date_range is not None:
            d_lo, d_hi = date_range
            or_df = or_df[(or_df.index >= d_lo) & (or_df.index <= d_hi)]
            if or_df.empty:
                continue

        common = or_df.index.intersection(daily.index)
        if len(common) == 0:
            continue
        or_df = or_df.loc[common]
        dsub  = daily.loc[common]

        atr      = or_df["atr_prev"].to_numpy(dtype=np.float64)
        or_range = or_df["or_range"].to_numpy(dtype=np.float64)

        with np.errstate(divide="ignore", invalid="ignore"):
            or_atr = np.where(atr > 0, or_range / atr, np.nan)

        valid = np.isfinite(atr) & (atr > 0)
        valid &= np.isfinite(or_atr) & (or_atr >= lo_f) & (or_atr <= hi_f)

        if use_nr7:
            valid &= or_df["nr7"].fillna(True).to_numpy(dtype=bool)

        if rel_vol_min > 0:
            rv = or_df["rel_vol"].to_numpy(dtype=np.float64)
            valid &= np.isfinite(rv) & (rv >= rel_vol_min)

        d_open = dsub["open"].to_numpy(dtype=np.float64)
        d_avgvol = (dsub["avgvol14"].to_numpy(dtype=np.float64)
                    if "avgvol14" in dsub.columns
                    else np.full(len(dsub), np.nan))
        valid &= np.isfinite(d_open)   & (d_open   >= MIN_PRICE)
        valid &= np.isfinite(d_avgvol) & (d_avgvol >= MIN_AVG_VOL)
        valid &= (atr >= MIN_ATR)

        if not valid.any():
            continue

        sub = or_df.loc[valid]
        dsub_valid = dsub.loc[valid]

        sig = pd.DataFrame({
            "asset":            asset,
            "date":             sub.index.values,
            "direction_first":  sub["direction"].astype(int).to_numpy(),
            "or_high":          sub["or_high"].to_numpy(dtype=np.float64),
            "or_low":           sub["or_low"].to_numpy(dtype=np.float64),
            "or_range":         sub["or_range"].to_numpy(dtype=np.float64),
            "or_atr":           or_atr[valid],
            "atr":              sub["atr_prev"].to_numpy(dtype=np.float64),
            "gap":              sub["gap"].to_numpy(dtype=np.float64),
            "sma20":            sub["sma20"].to_numpy(dtype=np.float64),
            "rel_vol":          sub["rel_vol"].to_numpy(dtype=np.float64),
            "day_close":        dsub_valid["close"].to_numpy(dtype=np.float64),
        })
        signal_frames.append(sig)

    if not signal_frames:
        return pd.DataFrame(), pd.Series(dtype=float)

    sig_df = pd.concat(signal_frames, ignore_index=True)
    sig_df["date"] = pd.to_datetime(sig_df["date"])
    sig_df["rv_rank"] = sig_df.groupby("date")["rel_vol"].rank(
        ascending=False, method="first")
    sig_df = sig_df[sig_df["rv_rank"] <= top_n]
    if sig_df.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    # ── 2. Trade simulation ─────────────────────────────────────────────
    equity = INITIAL_EQUITY
    trades: list[dict] = []
    eq_hist: dict[pd.Timestamp, float] = {}

    for date, day_sigs in sig_df.groupby("date", sort=True):
        date_ts = pd.Timestamp(date)
        eq_hist[date_ts] = equity

        for sig in day_sigs.itertuples(index=False):
            asset = sig.asset
            info = asset_data[asset]
            exec_bars = info["exec_cache"].get(orb_min, {}).get(date_ts)
            if exec_bars is None or exec_bars.empty:
                continue

            first_dir = int(sig.direction_first)
            gap = float(sig.gap)
            sma20 = float(sig.sma20)
            day_close = float(sig.day_close)

            if direction_source == "first_bar":
                direction = first_dir
            elif direction_source == "gap":
                if not np.isfinite(gap) or gap == 0:
                    continue
                direction = 1 if gap > 0 else -1
                if direction != first_dir:
                    continue
            elif direction_source == "sma_trend":
                if not np.isfinite(sma20) or not np.isfinite(day_close):
                    continue
                direction = 1 if day_close > sma20 else -1
                if direction != first_dir:
                    continue
            else:
                raise ValueError(direction_source)

            if direction == 0:
                continue

            trigger = float(sig.or_high) if direction == 1 else float(sig.or_low)

            high_a   = exec_bars["high"].to_numpy(dtype=np.float64)
            low_a    = exec_bars["low"].to_numpy(dtype=np.float64)
            open_a   = exec_bars["open"].to_numpy(dtype=np.float64)
            close_a  = exec_bars["close"].to_numpy(dtype=np.float64)
            volume_a = exec_bars["volume"].to_numpy(dtype=np.float64)

            if direction == 1:
                hits = high_a >= trigger
            else:
                hits = low_a <= trigger
            if not hits.any():
                continue
            hit_idx = int(hits.argmax())

            if breakout_vol_mult > 0:
                if "vol_sma20" not in exec_bars.columns:
                    continue
                vsma_arr = exec_bars["vol_sma20"].to_numpy(dtype=np.float64)
                if not np.isfinite(vsma_arr[hit_idx]):
                    continue
                if volume_a[hit_idx] < breakout_vol_mult * vsma_arr[hit_idx]:
                    continue

            if entry_type == "stop":
                bar_open = open_a[hit_idx]
                raw_fill = max(trigger, bar_open) if direction == 1 \
                    else min(trigger, bar_open)
                entry_px = raw_fill + direction * SLIPPAGE_TICKS * tick
            elif entry_type == "close":
                entry_px = close_a[hit_idx] + direction * SLIPPAGE_TICKS * tick
            else:
                raise ValueError(entry_type)
            entry_bar_idx = hit_idx

            if use_vwap_filter:
                if "vwap" not in exec_bars.columns:
                    continue
                vwap_arr = exec_bars["vwap"].to_numpy(dtype=np.float64)
                if not np.isfinite(vwap_arr[hit_idx]):
                    continue
                if direction == 1 and entry_px < vwap_arr[hit_idx]:
                    continue
                if direction == -1 and entry_px > vwap_arr[hit_idx]:
                    continue

            if require_retest:
                if entry_bar_idx + 1 >= len(exec_bars):
                    continue
                post_low   = low_a[entry_bar_idx + 1:]
                post_high  = high_a[entry_bar_idx + 1:]
                post_close = close_a[entry_bar_idx + 1:]
                if direction == 1:
                    retest_mask = (post_low <= trigger) & (post_close > trigger)
                else:
                    retest_mask = (post_high >= trigger) & (post_close < trigger)
                if not retest_mask.any():
                    continue
                retest_pos = int(retest_mask.argmax())
                entry_px = post_close[retest_pos] + direction * SLIPPAGE_TICKS * tick
                entry_bar_idx = entry_bar_idx + 1 + retest_pos

            atr = float(sig.atr)
            if stop_type == "atr":
                stop_dist = stop_param * atr
            elif stop_type == "or_opposite":
                stop_dist = stop_param * float(sig.or_range)
            elif stop_type == "fixed":
                stop_dist = stop_param
            else:
                raise ValueError(stop_type)
            if stop_dist <= 0:
                continue
            stop_px = entry_px - direction * stop_dist

            risk_dollars = RISK_PER_TRADE * equity
            shares = risk_dollars / stop_dist
            if shares * entry_px > LEVERAGE_CAP * equity:
                shares = LEVERAGE_CAP * equity / entry_px
            if shares <= 0 or not np.isfinite(shares):
                continue

            if target_type == "eod":
                target_px = np.nan
            elif target_type == "or_mult":
                target_px = entry_px + direction * target_param * float(sig.or_range)
            elif target_type == "R_mult":
                target_px = entry_px + direction * target_param * stop_dist
            else:
                raise ValueError(target_type)

            if entry_bar_idx + 1 >= len(exec_bars):
                exit_px = close_a[entry_bar_idx]
                exit_reason = "eod"
            else:
                after_high = high_a[entry_bar_idx + 1:]
                after_low  = low_a[entry_bar_idx + 1:]
                n = len(after_high)

                if direction == 1:
                    stop_hits = after_low <= stop_px
                else:
                    stop_hits = after_high >= stop_px

                if not np.isnan(target_px):
                    if direction == 1:
                        tgt_hits = after_high >= target_px
                    else:
                        tgt_hits = after_low <= target_px
                    tgt_pos = int(tgt_hits.argmax()) if tgt_hits.any() else n
                else:
                    tgt_pos = n

                stop_pos = int(stop_hits.argmax()) if stop_hits.any() else n

                if tgt_pos < n and tgt_pos <= stop_pos:
                    exit_px = target_px - direction * SLIPPAGE_TICKS * tick
                    exit_reason = "target"
                elif stop_pos < n:
                    exit_px = stop_px - direction * SLIPPAGE_TICKS * tick
                    exit_reason = "stop"
                else:
                    exit_px = close_a[-1]
                    exit_reason = "eod"

            gross = direction * (exit_px - entry_px) * shares
            comm  = COMMISSION * shares * 2
            net   = gross - comm
            R     = direction * (exit_px - entry_px) / stop_dist

            equity += net
            trades.append({
                "date": date_ts, "asset": asset, "direction": direction,
                "rel_vol": sig.rel_vol, "or_atr": sig.or_atr,
                "trigger": trigger, "entry_px": entry_px,
                "stop_px": stop_px, "exit_px": exit_px,
                "exit_reason": exit_reason,
                "shares": shares, "net_pnl": net, "R": R,
            })

        eq_hist[date_ts] = equity

    eq_series = pd.Series(eq_hist, dtype=np.float64)
    if not isinstance(eq_series.index, pd.DatetimeIndex):
        eq_series.index = pd.to_datetime(eq_series.index)
    eq_series = eq_series.sort_index()
    eq_series.name = "equity"

    return pd.DataFrame(trades), eq_series


# ═════════════════════════════════════════════════════════════════════════════
# PARAMETER GRID
# ═════════════════════════════════════════════════════════════════════════════
GRID = {
    "orb_min":           [5, 15, 30, 45, 60],
    "direction_source":  ["first_bar", "gap", "sma_trend"],
    "or_size_filter":    [(0, np.inf), (0.3, 1.5), (0.5, 1.2), (0.0, 0.6)],
    "use_nr7":           [False, True],
    "rel_vol_min":       [0.0, 1.0, 1.5, 2.0],
    "top_n":             [3, 5, 10],
    "breakout_vol_mult": [0.0, 1.2, 1.5],
    "use_vwap_filter":   [False, True],
    "require_retest":    [False, True],
    "stop_type":         ["atr"],
    "stop_param":        [0.10, 0.25, 0.50],
    "target_type":       ["eod", "or_mult", "R_mult"],
    "target_param":      [0.5, 1.0, 1.5, 3.0],
    "entry_type":        ["stop", "close"],
}
keys = list(GRID.keys())
combos = list(itertools.product(*[GRID[k] for k in keys]))
valid = []
for c in combos:
    d = dict(zip(keys, c))
    if d["target_type"] == "eod" and d["target_param"] != 0.5:
        continue
    if d["stop_type"] == "fixed":
        continue
    valid.append(d)

print(f"Grid: {len(combos):,} raw → {len(valid):,} valid")
if len(valid) > MAX_CONFIGS:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(valid), MAX_CONFIGS, replace=False)
    valid = [valid[i] for i in idx]
    print(f"Stratified sample: {len(valid):,} configs")


# ═════════════════════════════════════════════════════════════════════════════
# IN-SAMPLE / OUT-OF-SAMPLE SPLIT
# ═════════════════════════════════════════════════════════════════════════════
all_dates = sorted(set().union(*[
    info["or_cache"][5].index
    for info in asset_data.values()
    if 5 in info["or_cache"]
]))
split_idx  = int(len(all_dates) * WF_TRAIN_FRAC)
split_date = all_dates[split_idx]
in_range   = (all_dates[0], split_date)
oos_range  = (split_date + pd.Timedelta(days=1), all_dates[-1])

print(f"\nIS ends {split_date.date()}  |  OOS starts {oos_range[0].date()}\n")


# ═════════════════════════════════════════════════════════════════════════════
# IN-SAMPLE GRID SEARCH
# ═════════════════════════════════════════════════════════════════════════════
print("In-sample grid search ...")
results = []
skips = {"exception": 0, "no_signals": 0, "few_trades": 0, "no_variance": 0}
exception_samples: list[str] = []
t0 = time.time()

for i, params in enumerate(valid):
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(valid)}  ({time.time()-t0:.0f}s)")

    try:
        trades, equity = backtest_portfolio(asset_data, **params, date_range=in_range)
    except Exception as e:
        skips["exception"] += 1
        if len(exception_samples) < 5:
            exception_samples.append(f"{type(e).__name__}: {e} | params={params}")
        continue

    if trades.empty:
        skips["no_signals"] += 1
        continue
    if len(trades) < MIN_TRADES_IN:
        skips["few_trades"] += 1
        continue

    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        skips["no_variance"] += 1
        continue

    results.append({
        **{k: v for k, v in params.items() if k != "or_size_filter"},
        "or_size_lo": params["or_size_filter"][0],
        "or_size_hi": params["or_size_filter"][1],
        "in_trades":  len(trades),
        "in_sharpe":  float(rets.mean() / rets.std() * np.sqrt(252)),
        "in_return":  float(equity.iloc[-1] / INITIAL_EQUITY - 1),
        "in_hit":     float((trades["R"] > 0).mean()),
        "in_avgR":    float(trades["R"].mean()),
    })

res_df = pd.DataFrame(results)
if res_df.empty:
    raise RuntimeError(f"Zero configs survived IS. Skip counts: {skips}")

res_df = res_df.sort_values("in_sharpe", ascending=False).reset_index(drop=True)

print(f"\nIn-sample summary:")
print(f"  evaluated:              {len(res_df):,} / {len(valid):,}")
print(f"  skipped — exception:    {skips['exception']}")
print(f"  skipped — no signals:   {skips['no_signals']}")
print(f"  skipped — few trades:   {skips['few_trades']}")
print(f"  skipped — no variance:  {skips['no_variance']}")
if exception_samples:
    print("\nFirst exceptions:")
    for s in exception_samples:
        print(f"  {s}")

print("\n" + "=" * 90)
print("TOP 15 IN-SAMPLE CONFIGS")
print("=" * 90)
show_cols = ["orb_min","direction_source","rel_vol_min","top_n",
             "breakout_vol_mult","use_vwap_filter","require_retest",
             "stop_param","target_type","target_param","entry_type",
             "in_trades","in_sharpe","in_return","in_hit","in_avgR"]
display(res_df[show_cols].head(15).round(3))


# ═════════════════════════════════════════════════════════════════════════════
# OUT-OF-SAMPLE VALIDATION
# ═════════════════════════════════════════════════════════════════════════════
print("\nOut-of-sample validation on top 50 IS configs ...")
oos_results = []
oos_skips = {"exception": 0, "no_signals": 0, "few_trades": 0, "no_variance": 0}

for _, row in res_df.head(50).iterrows():
    params = {
        "orb_min":           int(row["orb_min"]),
        "direction_source":  row["direction_source"],
        "or_size_filter":    (float(row["or_size_lo"]), float(row["or_size_hi"])),
        "use_nr7":           bool(row["use_nr7"]),
        "rel_vol_min":       float(row["rel_vol_min"]),
        "top_n":             int(row["top_n"]),
        "breakout_vol_mult": float(row["breakout_vol_mult"]),
        "use_vwap_filter":   bool(row["use_vwap_filter"]),
        "require_retest":    bool(row["require_retest"]),
        "stop_type":         row["stop_type"],
        "stop_param":        float(row["stop_param"]),
        "target_type":       row["target_type"],
        "target_param":      float(row["target_param"]),
        "entry_type":        row["entry_type"],
    }
    try:
        trades, equity = backtest_portfolio(asset_data, **params, date_range=oos_range)
    except Exception as e:
        oos_skips["exception"] += 1
        print(f"    OOS EXCEPTION: {type(e).__name__}: {e}")
        continue

    if trades.empty:
        oos_skips["no_signals"] += 1
        continue
    if len(trades) < MIN_TRADES_OOS:
        oos_skips["few_trades"] += 1
        continue

    rets = equity.pct_change(fill_method=None).dropna()
    if len(rets) < 2 or rets.std() == 0:
        oos_skips["no_variance"] += 1
        continue

    oos_results.append({
        **params, "or_size_filter": str(params["or_size_filter"]),
        "oos_trades": len(trades),
        "oos_sharpe": float(rets.mean() / rets.std() * np.sqrt(252)),
        "oos_return": float(equity.iloc[-1] / INITIAL_EQUITY - 1),
        "oos_hit":    float((trades["R"] > 0).mean()),
        "oos_avgR":   float(trades["R"].mean()),
    })

print(f"\nOOS summary:")
print(f"  evaluated:              {len(oos_results):,} / 50")
print(f"  skipped — exception:    {oos_skips['exception']}")
print(f"  skipped — no signals:   {oos_skips['no_signals']}")
print(f"  skipped — few trades:   {oos_skips['few_trades']}")
print(f"  skipped — no variance:  {oos_skips['no_variance']}")

oos_df = pd.DataFrame(oos_results)
if oos_df.empty:
    raise RuntimeError(
        f"Zero configs survived OOS. Skips: {oos_skips}. "
        "Lower MIN_TRADES_OOS or expand the top-50 cut."
    )
oos_df = oos_df.sort_values("oos_sharpe", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("TOP 15 OUT-OF-SAMPLE CONFIGS")
print("=" * 90)
show_oos = ["orb_min","direction_source","rel_vol_min","top_n",
            "use_vwap_filter","require_retest","stop_param",
            "target_type","target_param","entry_type",
            "oos_trades","oos_sharpe","oos_return","oos_hit","oos_avgR"]
display(oos_df[show_oos].head(15).round(3))


# ═════════════════════════════════════════════════════════════════════════════
# BEST CONFIG → FULL SAMPLE
# ═════════════════════════════════════════════════════════════════════════════
best = oos_df.iloc[0]
print("\n" + "=" * 90); print("BEST CONFIG"); print("=" * 90)
for k, v in best.items():
    if k != "or_size_filter":
        print(f"  {k:>20}: {v}")

best_params = {
    "orb_min":           int(best["orb_min"]),
    "direction_source":  best["direction_source"],
    "or_size_filter":    eval(best["or_size_filter"]),
    "use_nr7":           bool(best["use_nr7"]),
    "rel_vol_min":       float(best["rel_vol_min"]),
    "top_n":             int(best["top_n"]),
    "breakout_vol_mult": float(best["breakout_vol_mult"]),
    "use_vwap_filter":   bool(best["use_vwap_filter"]),
    "require_retest":    bool(best["require_retest"]),
    "stop_type":         best["stop_type"],
    "stop_param":        float(best["stop_param"]),
    "target_type":       best["target_type"],
    "target_param":      float(best["target_param"]),
    "entry_type":        best["entry_type"],
}

trades_full, equity_full = backtest_portfolio(asset_data, **best_params, date_range=None)


# ═════════════════════════════════════════════════════════════════════════════
# PERFORMANCE METRICS
# ═════════════════════════════════════════════════════════════════════════════
def perf_stats(equity, trades, name):
    eq = equity.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if eq.empty or len(eq) < 2:
        return {"strategy": name}
    rets = eq.pct_change(fill_method=None).dropna()
    years = len(eq) / 252.0
    total = eq.iloc[-1] / INITIAL_EQUITY - 1
    cagr = (eq.iloc[-1] / INITIAL_EQUITY) ** (1 / years) - 1 \
        if years > 0 and eq.iloc[-1] > 0 else np.nan
    vol = rets.std() * np.sqrt(252) if len(rets) else np.nan
    sharpe = (rets.mean() * 252) / vol if vol and vol > 0 else np.nan
    mdd = (eq / eq.cummax() - 1).min()
    hit = (trades["R"] > 0).mean() if len(trades) else np.nan
    avgR = trades["R"].mean() if len(trades) else np.nan
    return {"strategy": name, "trades": len(trades),
            "total_return_%": total * 100, "CAGR_%": cagr * 100,
            "vol_%": vol * 100, "sharpe": sharpe,
            "hit_%": hit * 100, "avgR": avgR, "MDD_%": mdd * 100}

full_stats = perf_stats(equity_full, trades_full, "BEST ORB Portfolio")
print("\n" + "=" * 90); print("FULL-SAMPLE PERFORMANCE"); print("=" * 90)
display(pd.DataFrame([full_stats]).set_index("strategy").round(3))


# ═════════════════════════════════════════════════════════════════════════════
# BUY & HOLD — SPY ONLY
# ═════════════════════════════════════════════════════════════════════════════
if BENCHMARK not in asset_data:
    raise RuntimeError(f"Benchmark '{BENCHMARK}' missing from preprocessed assets.")

spy_close = asset_data[BENCHMARK]["df_rth"].groupby("date")["close"].last().dropna()
spy_close = spy_close[spy_close > 0]

spy_shares = INITIAL_EQUITY / spy_close.iloc[0]
bh_equity = (spy_close * spy_shares)
bh_equity.name = f"Buy & Hold ({BENCHMARK})"

common_idx = equity_full.index.intersection(bh_equity.index)
if len(common_idx) == 0:
    raise RuntimeError("Strategy and benchmark share no common dates.")

eq_plot = equity_full.reindex(common_idx)
bh_plot = bh_equity.reindex(common_idx)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bh_plot.index, y=bh_plot.values,
    name=f"Buy & Hold ({BENCHMARK})",
    line=dict(color="red", width=1.5, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=eq_plot.index, y=eq_plot.values,
    name=(f"BEST ORB ({best['orb_min']}m, {best['direction_source']}, "
          f"top-{best['top_n']} RV, "
          f"{'retest' if best['require_retest'] else 'direct'}, "
          f"stop {best['stop_param']}×ATR, "
          f"target {best['target_type']}({best['target_param']}))"),
    line=dict(color="blue", width=2.5),
))
fig.update_layout(
    title=(f"ORB Bootstrap — ETF universe (NQ excluded) vs {BENCHMARK} buy & hold"
           f"<br><sup>Universe: {', '.join(sorted(asset_data.keys()))}</sup>"),
    xaxis_title="Date", yaxis_title="Equity ($)",
    template="plotly_white", height=700, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(tickformat="$,.0f")
fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS
# ═════════════════════════════════════════════════════════════════════════════
if not trades_full.empty:
    print("\nExit-reason split:")
    display(trades_full["exit_reason"].value_counts())
    print("\nR distribution by exit reason:")
    display(trades_full.groupby("exit_reason")["R"]
                    .agg(["count", "mean", "std"]).round(3))
    print("\nDirection split (1=long, -1=short):")
    display(trades_full["direction"].value_counts())
    print("\nPer-asset trade count and PnL:")
    display(trades_full.groupby("asset").agg(
        trades=("net_pnl", "count"),
        total_pnl=("net_pnl", "sum"),
        avgR=("R", "mean"),
        hit=("R", lambda s: (s > 0).mean()),
    ).round(3).sort_values("total_pnl", ascending=False))

print(f"\n✓ Bootstrap complete.")
print(f"  Trading universe:           {sorted(asset_data.keys())}")
print(f"  Benchmark:                  {BENCHMARK}")
print(f"  IS configs evaluated:       {len(res_df):,}")
print(f"  OOS configs evaluated:      {len(oos_df):,}")
print(f"  Best OOS Sharpe:            {best['oos_sharpe']:.3f}")
print(f"  Best full-sample Sharpe:    {full_stats.get('sharpe', np.nan):.3f}")
print(f"  Best full-sample return:    {full_stats.get('total_return_%', np.nan):.1f}%")
print(f"  {BENCHMARK} B&H return:           "
      f"{(bh_equity.iloc[-1]/INITIAL_EQUITY - 1)*100:.1f}%")

Using `frames` from prior cell (11 assets).
Excluded: ['NQ']
Trading universe: ['CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT']

Preprocessing assets ...
   CPER: 109,059 RTH bars, 3,315 sessions
    DIA: 2,067,170 RTH bars, 5,966 sessions
    GLD: 1,924,936 RTH bars, 5,490 sessions
    IEF: 1,560,831 RTH bars, 5,966 sessions
    IWM: 2,217,669 RTH bars, 5,966 sessions
    QQQ: 2,245,015 RTH bars, 5,968 sessions
    SHY: 1,523,666 RTH bars, 5,966 sessions
    SLV: 1,752,691 RTH bars, 5,128 sessions
    SPY: 2,250,959 RTH bars, 5,968 sessions
    TLT: 2,005,004 RTH bars, 5,966 sessions

Preprocessed 10 assets: ['CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT']

Grid: 1,244,160 raw → 933,120 valid
Stratified sample: 800 configs

IS ends 2017-03-22  |  OOS starts 2017-03-23

In-sample grid search ...
  100/800  (67s)
  200/800  (118s)
  300/800  (146s)
  400/800  (199s)
  500/800  (228s)
  600/800  (259s)
  700/800  (305s)
  800/800  (335s)

In-

,orb_min,direction_source,rel_vol_min,top_n,breakout_vol_mult,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,in_trades,in_sharpe,in_return,in_hit,in_avgR
0,5,sma_trend,0.0,5,1.2,False,True,0.50,R_mult,1.0,stop,11,3.746,0.067,0.818,0.606
1,5,sma_trend,2.0,10,1.2,False,False,0.50,R_mult,3.0,stop,9,3.452,0.058,0.667,0.649
2,5,sma_trend,1.5,3,1.2,True,True,0.25,R_mult,1.5,stop,9,2.977,0.036,0.667,0.465
3,60,sma_trend,0.0,5,0.0,True,False,0.25,eod,0.5,close,6151,2.701,5308082.380,0.489,0.301
4,15,sma_trend,0.0,5,0.0,False,False,0.25,R_mult,3.0,close,7771,2.480,7813274.510,0.440,0.252
5,5,sma_trend,1.0,3,1.5,True,False,0.50,or_mult,1.5,stop,8,2.403,0.038,0.750,0.492
6,5,sma_trend,2.0,3,0.0,False,True,0.50,R_mult,1.5,stop,540,2.389,1.979,0.589,0.222
7,30,sma_trend,0.0,10,1.5,True,False,0.50,or_mult,1.5,close,4762,2.195,249.849,0.592,0.132
8,15,sma_trend,0.0,5,0.0,False,True,0.50,or_mult,1.5,close,7076,2.140,619.412,0.620,0.107
9,30,sma_trend,2.0,3,1.2,True,False,0.50,R_mult,1.0,stop,184,2.093,0.388,0.625,0.195



Out-of-sample validation on top 50 IS configs ...

OOS summary:
  evaluated:              46 / 50
  skipped — exception:    0
  skipped — no signals:   0
  skipped — few trades:   4
  skipped — no variance:  0

TOP 15 OUT-OF-SAMPLE CONFIGS


,orb_min,direction_source,rel_vol_min,top_n,use_vwap_filter,require_retest,stop_param,target_type,target_param,entry_type,oos_trades,oos_sharpe,oos_return,oos_hit,oos_avgR
0,30,sma_trend,2.0,3,True,False,0.50,R_mult,1.0,stop,89,3.824,0.406,0.674,0.397
1,15,sma_trend,0.0,5,False,False,0.25,R_mult,3.0,close,3760,2.682,2890.493,0.454,0.254
2,30,sma_trend,1.5,3,False,False,0.25,or_mult,1.5,close,118,2.621,0.756,0.441,0.572
3,5,sma_trend,2.0,3,False,True,0.50,R_mult,0.5,close,80,2.462,0.175,0.800,0.211
4,45,sma_trend,1.0,3,True,False,0.50,R_mult,3.0,close,715,2.217,3.936,0.583,0.237
5,30,sma_trend,1.5,5,True,False,0.25,or_mult,3.0,close,261,2.203,1.704,0.425,0.480
6,60,sma_trend,1.0,3,True,False,0.50,R_mult,1.5,close,796,2.132,2.884,0.585,0.182
7,30,sma_trend,0.0,10,True,False,0.50,or_mult,1.5,close,1870,2.131,9.817,0.591,0.139
8,30,sma_trend,1.0,10,False,False,0.50,eod,0.5,close,757,2.126,5.021,0.547,0.252
9,15,sma_trend,0.0,5,False,True,0.25,or_mult,1.0,close,212,2.096,0.686,0.448,0.291



BEST CONFIG
               orb_min: 30
      direction_source: sma_trend
               use_nr7: False
           rel_vol_min: 2.0
                 top_n: 3
     breakout_vol_mult: 1.2
       use_vwap_filter: True
        require_retest: False
             stop_type: atr
            stop_param: 0.5
           target_type: R_mult
          target_param: 1.0
            entry_type: stop
            oos_trades: 89
            oos_sharpe: 3.823731385253347
            oos_return: 0.40615126645800514
               oos_hit: 0.6741573033707865
              oos_avgR: 0.3968586469024315

FULL-SAMPLE PERFORMANCE


,trades,total_return_%,CAGR_%,vol_%,sharpe,hit_%,avgR,MDD_%
strategy,,,,,,,,
BEST ORB Portfolio,273,95.119,25.182,8.585,2.7,64.103,0.261,-6.206


... (5 outputs left out)

direction
-1    153
 1    120
Name: count, dtype: int64


Per-asset trade count and PnL:


,trades,total_pnl,avgR,hit
asset,,,,
GLD,77,9219.264,0.376,0.662
DIA,46,3228.484,0.213,0.630
SLV,19,2705.714,0.348,0.737
IWM,33,2595.490,0.248,0.636
QQQ,27,2231.119,0.234,0.667
TLT,42,2104.663,0.176,0.595
SPY,25,1827.675,0.164,0.600
IEF,4,-132.763,-0.033,0.500



✓ Bootstrap complete.
  Trading universe:           ['CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT']
  Benchmark:                  SPY
  IS configs evaluated:       724
  OOS configs evaluated:      46
  Best OOS Sharpe:            3.824
  Best full-sample Sharpe:    2.700
  Best full-sample return:    95.1%
  SPY B&H return:           1159.0%


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB TIMEFRAME SELECTION VIA TIME-SERIES FEATURES — NQ LOCAL DATA
#
# Fixes accumulated across iterations:
#   1. atr_prev propagated into or_df (fixes "No trades produced" error)
#   2. r_matrix reindexed to guarantee all OR_WINDOWS columns exist
#   3. best_tf NaNs dropped before int cast
#   4. Classifier trained ONLY on sessions where ≥2 OR timeframes traded
#      (fixes the "constant prediction of 5m" problem)
#   5. y_pred_train / y_pred_test wrapped in pd.Series with DatetimeIndex
#      (fixes the AttributeError: 'numpy.ndarray' object has no attribute 'index')
#   6. Added ORACLE upper-bound equity curve for honest comparison
#   7. Fallback policy uses best AVAILABLE TF when predicted TF didn't trade
#   8. VWAP and vol_sma20 precomputed on df_rth to support optional filters
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ───────────────────────────────────────────────────────────
FILEPATH         = "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv"
TZ               = "America/New_York"

INITIAL_EQUITY   = 25_000.0
RISK_PER_TRADE   = 0.01
LEVERAGE_CAP     = 4.0
COMMISSION       = 0.0035
SLIPPAGE_TICKS   = 1
TICK             = 0.25

RTH_START        = pd.Timestamp("09:30").time()
RTH_END          = pd.Timestamp("16:00").time()
REL_VOL_LOOKBACK = 14
ATR_LOOKBACK     = 14

BEST_CONFIG = {
    "orb_min":           5,
    "direction_source":  "first_bar",
    "rel_vol_min":       1.5,
    "breakout_vol_mult": 0.0,
    "use_vwap_filter":   False,
    "require_retest":    True,
    "stop_type":         "atr",
    "stop_param":        0.25,
    "target_type":       "R_mult",
    "target_param":      1.5,
    "entry_type":        "close",
}

OR_WINDOWS = [5, 15, 30, 45, 60]
WF_TRAIN_FRAC = 0.60

# ── Load local 1-minute NQ data ─────────────────────────────────────────────
raw = pd.read_csv(FILEPATH)
raw.columns = [c.lower() for c in raw.columns]
dt_col = next((c for c in ("datetime","timestamp","ts_event","date") if c in raw.columns), None)
if dt_col is None:
    raise ValueError(f"No datetime column. Headers: {list(raw.columns)}")

raw[dt_col] = pd.to_datetime(raw[dt_col], utc=False, errors="coerce")
if raw[dt_col].dt.tz is None:
    raw[dt_col] = raw[dt_col].dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT")
raw = raw.dropna(subset=[dt_col]).set_index(dt_col).sort_index()

keep = [c for c in ("open","high","low","close","volume") if c in raw.columns]
df_1m = raw[keep].apply(pd.to_numeric, errors="coerce").dropna(subset=["open","high","low","close"])
df_1m = df_1m.tz_convert(TZ)
df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()

times = df_1m.index.time
rth_mask = (times >= RTH_START) & (times < RTH_END)
df_rth = df_1m[rth_mask].copy()
df_rth["date"] = df_rth.index.normalize().tz_localize(None)

# Precompute session VWAP and 20-bar volume SMA
tp = (df_rth["high"] + df_rth["low"] + df_rth["close"]) / 3
df_rth["vwap"] = (
    (tp * df_rth["volume"]).groupby(df_rth["date"]).cumsum()
    / df_rth["volume"].groupby(df_rth["date"]).cumsum().replace(0, np.nan)
)
df_rth["vol_sma20"] = df_rth["volume"].rolling(20, min_periods=20).mean()

print(f"Loaded NQ 1m: {len(df_1m):,} bars  →  RTH: {len(df_rth):,} bars, "
      f"{df_rth['date'].nunique():,} sessions")

# ── Daily bars & features ──────────────────────────────────────────────────
AGG = {"open":"first","high":"max","low":"min","close":"last","volume":"sum"}
daily = df_rth.resample("1D").agg(AGG).dropna(subset=["open","high","low","close"])
if daily.index.tz is not None:
    daily.index = daily.index.tz_localize(None)

prev_close = daily["close"].shift(1)
tr = pd.concat([
    daily["high"] - daily["low"],
    (daily["high"] - prev_close).abs(),
    (daily["low"]  - prev_close).abs(),
], axis=1).max(axis=1)
daily["atr14"]    = tr.ewm(alpha=1/ATR_LOOKBACK, adjust=False, min_periods=ATR_LOOKBACK).mean()
daily["atr_prev"] = daily["atr14"].shift(1)
daily["sma20"]    = daily["close"].rolling(20, min_periods=20).mean()
daily["range"]    = daily["high"] - daily["low"]
daily["nr7"]      = daily["range"] == daily["range"].rolling(7, min_periods=7).min()
daily["gap"]      = daily["open"] - prev_close
daily["gap_pct"]  = daily["gap"] / prev_close
daily["avgvol14"] = daily["volume"].rolling(14, min_periods=14).mean()

bb_mid = daily["close"].rolling(20, min_periods=20).mean()
bb_std = daily["close"].rolling(20, min_periods=20).std()
daily["bb_width"] = (4 * bb_std) / bb_mid

ret_1d = daily["close"].pct_change()
daily["rv5"]  = ret_1d.rolling(5, min_periods=5).std() * np.sqrt(252)
daily["rv20"] = ret_1d.rolling(20, min_periods=20).std() * np.sqrt(252)
daily["rv_ratio"] = daily["rv5"] / daily["rv20"].replace(0, np.nan)

daily["atr_sma20"] = daily["atr14"].rolling(20, min_periods=20).mean()
daily["atr_ratio"] = daily["atr14"] / daily["atr_sma20"].replace(0, np.nan)
daily["prior_range_atr"] = daily["range"].shift(1) / daily["atr_prev"].replace(0, np.nan)


def hurst_rs(series: np.ndarray) -> float:
    n = len(series)
    if n < 8:
        return np.nan
    max_lag = max(3, n // 2)
    valid_lags, tau = [], []
    for lag in range(2, max_lag + 1):
        if n - lag < 2:
            continue
        diff = series[lag:] - series[:-lag]
        s = float(np.std(diff))
        if s > 0 and np.isfinite(s):
            valid_lags.append(lag)
            tau.append(s)
    if len(valid_lags) < 2:
        return np.nan
    return float(np.polyfit(np.log(valid_lags), np.log(tau), 1)[0])


log_ret = np.log(daily["close"] / daily["close"].shift(1))
daily["hurst20"] = log_ret.rolling(20, min_periods=20).apply(
    lambda x: hurst_rs(np.asarray(x, dtype=float)), raw=False)
daily["dow"] = daily.index.dayofweek

FEAT_COLS = ["gap_pct","atr_ratio","bb_width","rv_ratio",
             "prior_range_atr","hurst20","dow"]
PROPAGATE_COLS = list(set(FEAT_COLS + ["atr_prev","nr7","sma20","gap"]))

print("\nFeature coverage:")
for c in FEAT_COLS:
    print(f"  {c:>20}: {daily[c].notna().sum():,} / {len(daily):,}")

# ── OR caches ──────────────────────────────────────────────────────────────
or_cache: dict[int, pd.DataFrame] = {}
exec_cache: dict[int, dict] = {}

for orb_min in OR_WINDOWS:
    rows = []
    exec_cache[orb_min] = {}
    for date, sess in df_rth.groupby("date", sort=True):
        if len(sess) < orb_min + 1:
            continue
        or_bars = sess.iloc[:orb_min]
        o = or_bars["open"].iloc[0]
        c = or_bars["close"].iloc[-1]
        h = or_bars["high"].max()
        l = or_bars["low"].min()
        v = or_bars["volume"].sum()
        rows.append({"date": date, "or_open": o, "or_close": c,
                     "or_high": h, "or_low": l, "or_vol": v,
                     "or_range": h - l,
                     "direction": 1 if c > o else (-1 if c < o else 0)})
        exec_cache[orb_min][date] = sess.iloc[orb_min:]

    or_df = pd.DataFrame(rows).set_index("date")
    or_df["rel_vol"] = or_df["or_vol"] / (
        or_df["or_vol"].rolling(REL_VOL_LOOKBACK, min_periods=REL_VOL_LOOKBACK)
                .mean().shift(1))
    for col in PROPAGATE_COLS:
        if col in daily.columns:
            or_df[col] = daily[col].reindex(or_df.index)
    or_cache[orb_min] = or_df


# ── Backtest engine ────────────────────────────────────────────────────────
def backtest_single_day(orb_min, date, row, exec_bars, config):
    atr = row.get("atr_prev", np.nan)
    if pd.isna(atr) or atr <= 0:
        return None
    rv = row.get("rel_vol", np.nan)
    if config["rel_vol_min"] > 0 and (pd.isna(rv) or rv < config["rel_vol_min"]):
        return None
    direction = int(row["direction"])
    if direction == 0:
        return None
    trigger = row["or_high"] if direction == 1 else row["or_low"]
    hit = (exec_bars["high"].to_numpy() >= trigger) if direction == 1 \
        else (exec_bars["low"].to_numpy() <= trigger)
    if not hit.any():
        return None
    hit_idx = int(np.argmax(hit))
    bar = exec_bars.iloc[hit_idx]
    entry_px = bar["close"] + direction * SLIPPAGE_TICKS * TICK

    if config.get("use_vwap_filter", False):
        vwap = bar.get("vwap", np.nan)
        if pd.isna(vwap):
            return None
        if direction == 1 and entry_px < vwap:
            return None
        if direction == -1 and entry_px > vwap:
            return None

    if config.get("breakout_vol_mult", 0) > 0:
        vsma = bar.get("vol_sma20", np.nan)
        if pd.isna(vsma) or bar["volume"] < config["breakout_vol_mult"] * vsma:
            return None

    if config.get("require_retest", False):
        post = exec_bars.iloc[hit_idx + 1:]
        retested = False
        for pos, (bidx, b) in enumerate(post.iterrows()):
            if direction == 1:
                if b["low"] <= trigger and b["close"] > trigger:
                    retested = True
                    entry_px = b["close"] + direction * SLIPPAGE_TICKS * TICK
                    hit_idx = pos
                    bar = b
                    break
            else:
                if b["high"] >= trigger and b["close"] < trigger:
                    retested = True
                    entry_px = b["close"] + direction * SLIPPAGE_TICKS * TICK
                    hit_idx = pos
                    bar = b
                    break
        if not retested:
            return None
        exec_bars = post

    if config["stop_type"] == "atr":
        stop_dist = config["stop_param"] * atr
    elif config["stop_type"] == "or_opposite":
        stop_dist = config["stop_param"] * row["or_range"]
    else:
        stop_dist = config["stop_param"]
    if stop_dist <= 0:
        return None
    stop_px = entry_px - direction * stop_dist

    if config["target_type"] == "eod":
        target_px = np.nan
    elif config["target_type"] == "or_mult":
        target_px = entry_px + direction * config["target_param"] * row["or_range"]
    else:
        target_px = entry_px + direction * config["target_param"] * stop_dist

    after = exec_bars.iloc[hit_idx + 1:]
    exit_px = None
    exit_reason = None
    for _, b in after.iterrows():
        if direction == 1:
            if not np.isnan(target_px) and b["high"] >= target_px:
                exit_px = target_px - direction * SLIPPAGE_TICKS * TICK
                exit_reason = "target"
                break
            if b["low"] <= stop_px:
                exit_px = stop_px - direction * SLIPPAGE_TICKS * TICK
                exit_reason = "stop"
                break
        else:
            if not np.isnan(target_px) and b["low"] <= target_px:
                exit_px = target_px - direction * SLIPPAGE_TICKS * TICK
                exit_reason = "target"
                break
            if b["high"] >= stop_px:
                exit_px = stop_px - direction * SLIPPAGE_TICKS * TICK
                exit_reason = "stop"
                break

    if exit_px is None:
        exit_px = exec_bars["close"].iloc[-1]
        exit_reason = "eod"

    return {"date": date, "orb_min": orb_min,
            "R": direction * (exit_px - entry_px) / stop_dist}


# ── Run all timeframes ─────────────────────────────────────────────────────
print("\nRunning all ORB timeframes ...")
all_trades = []
for orb_min in OR_WINDOWS:
    n_tf = 0
    for date, row in or_cache[orb_min].iterrows():
        if date not in daily.index:
            continue
        exec_bars = exec_cache[orb_min].get(date)
        if exec_bars is None or exec_bars.empty:
            continue
        cfg = dict(BEST_CONFIG)
        cfg["orb_min"] = orb_min
        t = backtest_single_day(orb_min, date, row, exec_bars, cfg)
        if t is not None:
            all_trades.append(t)
            n_tf += 1
    print(f"  {orb_min:>3}m: {n_tf:,} trades")

trades_df = pd.DataFrame(all_trades)
if trades_df.empty:
    raise RuntimeError("No trades produced.")

r_matrix = trades_df.pivot_table(index="date", columns="orb_min",
                                 values="R", aggfunc="first")
r_matrix = r_matrix.reindex(columns=OR_WINDOWS)

print(f"\nR matrix: {r_matrix.shape}")
print("Per-timeframe non-NaN trade counts:")
print(r_matrix.notna().sum())

# ═════════════════════════════════════════════════════════════════════════════
# LABEL: only keep days where ≥2 timeframes traded (real choice available)
# ═════════════════════════════════════════════════════════════════════════════
n_available = r_matrix[OR_WINDOWS].notna().sum(axis=1)
print(f"\nSessions with 1 TF available:  {(n_available == 1).sum():,}")
print(f"Sessions with 2 TF available:  {(n_available == 2).sum():,}")
print(f"Sessions with 3+ TF available: {(n_available >= 3).sum():,}")

multi_tf_days = n_available[n_available >= 2].index
r_multi = r_matrix.loc[multi_tf_days].copy()
r_multi["best_tf"] = r_multi[OR_WINDOWS].idxmax(axis=1)
r_multi["best_R"] = r_multi[OR_WINDOWS].max(axis=1)

print(f"\nTraining days (≥2 TFs available): {len(r_multi):,}")
print("Label distribution on these days:")
print(r_multi["best_tf"].value_counts().sort_index())

# ═════════════════════════════════════════════════════════════════════════════
# FEATURE MATRIX — restricted to multi-TF days
# ═════════════════════════════════════════════════════════════════════════════
feat_df = daily[FEAT_COLS].dropna()
common_idx = feat_df.index.intersection(r_multi.index)
print(f"\nCommon index (features ∩ multi-TF days): {len(common_idx):,}")

if len(common_idx) < 50:
    raise RuntimeError(f"Too few overlapping days ({len(common_idx)}).")

X = feat_df.loc[common_idx]
y = r_multi.loc[common_idx, "best_tf"].astype(int)

print(f"Feature matrix: {X.shape[0]} days × {X.shape[1]} features")
print(f"Label distribution:\n{y.value_counts().sort_index()}")

# ═════════════════════════════════════════════════════════════════════════════
# WALK-FORWARD TRAIN / TEST
# Predictions wrapped in pd.Series with DatetimeIndex
# ═════════════════════════════════════════════════════════════════════════════
n = len(X)
split = int(n * WF_TRAIN_FRAC)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"\nTrain: {len(X_train):,} days  |  Test: {len(X_test):,} days")
print(f"Train label distribution:\n{y_train.value_counts().sort_index()}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

clf = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10,
    random_state=42, n_jobs=-1,
)
clf.fit(X_train_s, y_train)

# Wrap predictions in pd.Series with the DatetimeIndex
y_pred_train = pd.Series(clf.predict(X_train_s), index=X_train.index,
                          name="pred_tf")
y_pred_test  = pd.Series(clf.predict(X_test_s),  index=X_test.index,
                          name="pred_tf")

print(f"\nTrain accuracy: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_pred_test):.3f}")
print(f"\nTest prediction distribution:")
print(y_pred_test.value_counts().sort_index())

print("\nClassification report (test):")
print(classification_report(y_test, y_pred_test, zero_division=0))

imp = pd.Series(clf.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print("\nFeature importances:")
for k, v in imp.items():
    print(f"  {k:>20}: {v:.4f}")

# ═════════════════════════════════════════════════════════════════════════════
# EQUITY CURVES — evaluated on the FULL test window
# ═════════════════════════════════════════════════════════════════════════════
def pick_best_available(r_row, predicted_tf):
    """Return R of predicted_tf if available, else best available TF's R."""
    if predicted_tf in r_row.index and pd.notna(r_row[predicted_tf]):
        return r_row[predicted_tf]
    valid = r_row[OR_WINDOWS].dropna()
    return valid.max() if not valid.empty else 0.0


test_start = X_test.index.min()
test_end = X_test.index.max()
full_test_idx = r_matrix.loc[test_start:test_end].index

selected_R = []
for date in full_test_idx:
    r_row = r_matrix.loc[date]
    if date in y_pred_test.index:
        pred_tf = int(y_pred_test.loc[date])
    else:
        pred_tf = 5
    selected_R.append(pick_best_available(r_row, pred_tf))

selected_R = pd.Series(selected_R, index=full_test_idx)
selected_pnl = selected_R * (RISK_PER_TRADE * INITIAL_EQUITY)

static_R_test = r_matrix.loc[full_test_idx, 5].fillna(0)
static_pnl_test = static_R_test * (RISK_PER_TRADE * INITIAL_EQUITY)

oracle_R = r_matrix.loc[full_test_idx, OR_WINDOWS].max(axis=1).fillna(0)
oracle_pnl = oracle_R * (RISK_PER_TRADE * INITIAL_EQUITY)

bh_close_test = daily.loc[full_test_idx, "close"]
bh_equity_test = (bh_close_test / bh_close_test.iloc[0]) * INITIAL_EQUITY
bh_pnl_test = bh_equity_test.diff().fillna(0)

print(f"\nTest period: {test_start.date()} → {test_end.date()}  "
      f"({len(full_test_idx)} sessions)")


def perf_stats(pnl_series, name):
    eq = INITIAL_EQUITY + pnl_series.cumsum()
    rets = eq.pct_change(fill_method=None).dropna()
    years = len(eq) / 252.0
    total = eq.iloc[-1] / INITIAL_EQUITY - 1
    cagr = (eq.iloc[-1] / INITIAL_EQUITY) ** (1 / years) - 1 \
        if years > 0 and eq.iloc[-1] > 0 else np.nan
    vol = rets.std() * np.sqrt(252) if len(rets) else np.nan
    sharpe = (rets.mean() * 252) / vol if vol and vol > 0 else np.nan
    mdd = (eq / eq.cummax() - 1).min() if len(eq) else np.nan
    hit = (pnl_series > 0).mean()
    avg_R = (pnl_series / (RISK_PER_TRADE * INITIAL_EQUITY)).mean()
    n_trades = (pnl_series != 0).sum()
    return {"strategy": name, "trades": int(n_trades),
            "total_return_%": total * 100,
            "CAGR_%": cagr * 100 if not np.isnan(cagr) else np.nan,
            "vol_%": vol * 100 if not np.isnan(vol) else np.nan,
            "sharpe": sharpe, "hit_%": hit * 100, "avgR": avg_R,
            "MDD_%": mdd * 100 if not np.isnan(mdd) else np.nan}


summary = pd.DataFrame([
    perf_stats(selected_pnl, "Feature-Selected ORB"),
    perf_stats(static_pnl_test, "Static 5m ORB (best config)"),
    perf_stats(oracle_pnl, "Oracle (upper bound)"),
    perf_stats(bh_pnl_test, "Buy & Hold (NQ)"),
]).set_index("strategy")

print("\n" + "=" * 90)
print("OUT-OF-SAMPLE PERFORMANCE COMPARISON")
print("=" * 90)
display(summary.round(2))

# ═════════════════════════════════════════════════════════════════════════════
# EQUITY CURVES PLOT
# ═════════════════════════════════════════════════════════════════════════════
def build_equity(pnl, name):
    eq = INITIAL_EQUITY + pnl.cumsum()
    eq.name = name
    return eq


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bh_equity_test.index, y=bh_equity_test.values,
    name="Buy & Hold (NQ)",
    line=dict(color="red", width=1.5, dash="dot")))
fig.add_trace(go.Scatter(
    x=build_equity(static_pnl_test, "static").index,
    y=build_equity(static_pnl_test, "static").values,
    name=f"Static 5m ORB "
         f"(Sharpe={summary.loc['Static 5m ORB (best config)', 'sharpe']:.2f})",
    line=dict(color="orange", width=2)))
fig.add_trace(go.Scatter(
    x=build_equity(selected_pnl, "selected").index,
    y=build_equity(selected_pnl, "selected").values,
    name=f"Feature-Selected ORB "
         f"(Sharpe={summary.loc['Feature-Selected ORB', 'sharpe']:.2f})",
    line=dict(color="blue", width=2.5)))
fig.add_trace(go.Scatter(
    x=build_equity(oracle_pnl, "oracle").index,
    y=build_equity(oracle_pnl, "oracle").values,
    name=f"Oracle upper bound "
         f"(Sharpe={summary.loc['Oracle (upper bound)', 'sharpe']:.2f})",
    line=dict(color="green", width=1.5, dash="dash")))

fig.update_layout(
    title=("ORB Timeframe Selection — Corrected Label + Oracle Bound"
           "<br><sup>Classifier trained only on sessions with ≥2 ORB timeframes "
           "available; fallback uses best available TF</sup>"),
    xaxis_title="Date", yaxis_title="Equity ($)",
    template="plotly_white", height=700, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1))
fig.update_yaxes(tickformat="$,.0f")
fig.show()

# ═════════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS
# ═════════════════════════════════════════════════════════════════════════════
print("\nConfusion matrix (test set, multi-TF days only):")
display(pd.crosstab(y_test, y_pred_test, margins=True))

print("\nRealized avg R per predicted timeframe (OOS):")
for tf in sorted(y_pred_test.unique()):
    mask = y_pred_test == tf
    if mask.any():
        R_vals = selected_R.loc[y_pred_test.index[mask]] \
            if y_pred_test.index[mask].isin(selected_R.index).any() else pd.Series()
        if len(R_vals):
            print(f"  predicted {int(tf):>3}m: n={mask.sum():>4}  "
                  f"avgR={R_vals.mean():>6.3f}  "
                  f"hit={(R_vals > 0).mean()*100:>5.1f}%")

# Full-sample static 5m for reference
static_5m_full_R = r_matrix[5].fillna(0)
static_5m_full_pnl = static_5m_full_R * (RISK_PER_TRADE * INITIAL_EQUITY)
eq_static_full = build_equity(static_5m_full_pnl, "Static 5m Full")

print("\nFull-sample static 5m ORB (for reference):")
print(f"  total return: {(eq_static_full.iloc[-1]/INITIAL_EQUITY - 1)*100:.1f}%")
print(f"  final equity: ${eq_static_full.iloc[-1]:,.0f}")

print("\n✓ Feature engineering and timeframe selection complete.")

Loaded NQ 1m: 5,293,774 bars  →  RTH: 1,542,462 bars, 4,016 sessions

Feature coverage:
               gap_pct: 4,015 / 4,016
             atr_ratio: 3,984 / 4,016
              bb_width: 3,997 / 4,016
              rv_ratio: 3,996 / 4,016
       prior_range_atr: 4,002 / 4,016
               hurst20: 2,834 / 4,016
                   dow: 4,016 / 4,016

Running all ORB timeframes ...
    5m: 484 trades
   15m: 288 trades
   30m: 214 trades
   45m: 188 trades
   60m: 175 trades

R matrix: (669, 5)
Per-timeframe non-NaN trade counts:
orb_min
5     484
15    288
30    214
45    188
60    175
dtype: int64

Sessions with 1 TF available:  336
Sessions with 2 TF available:  144
Sessions with 3+ TF available: 189

Training days (≥2 TFs available): 333
Label distribution on these days:
best_tf
5     193
15     76
30     33
45     31
Name: count, dtype: int64

Common index (features ∩ multi-TF days): 179
Feature matrix: 179 days × 7 features
Label distribution:
best_tf
5     104
15     44
30     

,trades,total_return_%,CAGR_%,vol_%,sharpe,hit_%,avgR,MDD_%
strategy,,,,,,,,
Feature-Selected ORB,296,16.00,13.47,17.05,0.75,42.23,0.05,-17.78
Static 5m ORB (best config),241,19.03,15.99,15.33,0.96,35.47,0.06,-12.45
Oracle (upper bound),296,31.53,26.28,15.79,1.48,45.27,0.11,-15.93
Buy & Hold (NQ),295,269.27,204.10,2534783.24,0.30,56.08,0.91,-100.11



Confusion matrix (test set, multi-TF days only):


pred_tf,5,15,All
best_tf,,,
5,48,1,49
15,13,0,13
30,4,2,6
45,4,0,4
All,69,3,72



Realized avg R per predicted timeframe (OOS):
  predicted   5m: n=  69  avgR= 0.066  hit= 43.5%
  predicted  15m: n=   3  avgR=-0.670  hit=  0.0%

Full-sample static 5m ORB (for reference):
  total return: 35.5%
  final equity: $33,877

✓ Feature engineering and timeframe selection complete.


In [ ]:
# double down feature
# uses time series to find the highest probable days to double down on

In [ ]:
# volume profile (aggregated)


## Module.5
### ML/AI

In [ ]:
# forest tree classifiers

In [ ]:
# NN

## Module.6
### Prop firm optimization

In [ ]:
# load prop firm assets (topstep)

In [ ]:
# bootstrap strategy variations
# must adhere to prop firm rules & MLL

In [ ]:
# montecarlo simulate odds of passing

## Module.7
### Backtesting

In [ ]:
# rolling wfa

In [ ]:
# random wfa

In [ ]:
# t-test simulation

In [ ]:
# monte carlo simulation

In [ ]:
# regression testing (for randomness/noise)